# JVM and budget checks on DBR 19: the node-local native copy, the standard-access engine budget, Java hypot on x86

**What and why:** three checks the measurement pass's pool session carries, each by `MODE` (set it with `--set-var`):

- `MODE = "native_copy"` (dedicated): the executor JVMs load GDAL and PROJ from the node-local copy under
  `java.io.tmpdir/geobrix/native` (on DBR 19 `/local_disk0/tmp/geobrix/native/<version>-<jni>-<manifest>`), not from the
  notebook's environment; with `RACE = True` the environment's `share/proj` is deleted between passes (probe C), and the
  JVM companions `boundingbox` and `worldtorastercoordy` still return no NULL over `N_TILES` corpus tiles, with no executor lost.
- `MODE = "standard_budget"` (standard access): the driver carries the classic caps (read cap 1536 MiB decoded, stream
  cap 256 MiB), not Serverless's 64 MiB. Read by `raster_gbx` as a virtual and a materialized tile, a Float32 tile of
  `BUDGET_PX` (144 MiB decoded) and one of `BUDGET_LARGE_PX` (702 MiB) go through `gbx_rst_summary` and the GDAL engine's
  `gbx_rst_slope` with no refusal; one of `OVER_PX` (1600 MiB) is refused by the engine over the read cap in both forms;
  and `raster_gbx` refuses to materialize a file of `NOISE_PX` noise (295 MiB decoded, 263 MiB on disk) over the
  stream cap.
- `MODE = "java_hypot"` (dedicated, x86): Java `Math.hypot` on the driver and in the executors' generated code, and the
  Python port (`pyvx._jmath.hypot`) in a worker, give the bits of the 1164 vectors generated with Java 17 on arm64
  (`python/geobrix/test/pyvx/data/java_hypot.txt`); the TIN of both levels is then compared on random point sets,
  each triangle as its set of vertices (the levels may start a triangle's ring on a different vertex, so its WKB bytes
  can differ when the triangle does not).

**Expected output:** one JSON (`dbutils.notebook.exit`) with `MODE`, the node architecture and the mode's rows, and a
`verdict` PASS or FAIL.

**Expected time:** native_copy 5 to 10 min at 4000 tiles; standard_budget 6 to 12 min (the first run writes the tiles); java_hypot 2 to 4 min (after start).

**Monitoring:** one printed line per step; the job's time limit ends a stuck run; cancel from the Runs page. No retries.


In [ ]:
%pip install --force-reinstall --no-deps "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix/0.6.0/env6/geobrix-0.6.0+env6-py3-none-any.whl"
%pip install "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix/0.6.0/env6/geobrix-0.6.0+env6-py3-none-any.whl"


In [ ]:
%restart_python


In [ ]:
MODE = "java_hypot"  # native_copy | standard_budget | java_hypot
N_TILES = 4000
PASSES = 2
RACE = True
CORPUS = "/Volumes/geospatial_docs/geobrix/sample-data/bench-corpus/rows"
SCRATCH = "/Volumes/geospatial_docs/geobrix/sample-data/_probe/t25"
BUDGET_PX = 6144  # 6144 x 6144 Float32: 144 MiB decoded
BUDGET_LARGE_PX = 13568  # 702 MiB decoded: its slope (input and output) is within the 1536 MiB call budget
OVER_PX = 20480  # 1600 MiB decoded: over the 1536 MiB read cap
NOISE_PX = 8800  # noise: 295 MiB decoded, 263 MiB on disk, over raster_gbx's 256 MiB stream cap
TIN_SETS = 200


In [ ]:
import json
import os
import platform
import struct

RESULT = {"mode": MODE, "arch": platform.machine()}
print("MODE", MODE, "arch", RESULT["arch"], flush=True)


In [ ]:
if MODE == "native_copy":
    import shutil

    from pyspark.sql import functions as F

    from databricks.labs.gbx import _native
    from databricks.labs.gbx.manager import jvm

    NATIVE = _native.native_dir()
    PKG = "com.databricks.labs.gbx.rasterx.expressions."
    for name, fqn in {
        "gbx__l1_rst_fromcontent": PKG + "constructor.RST_FromContent",
        "gbx__l1_rst_boundingbox": PKG + "accessors.RST_BoundingBox",
        "gbx__l1_rst_worldtorastercoordy": PKG + "RST_WorldToRasterCoordY",
    }.items():
        why = jvm.install_private_l1(spark, fqn, name)
        if why:
            raise RuntimeError(why)

    def node_report(rows):
        import json as _j
        import os as _o
        import socket

        def ppid(pid):
            with open(f"/proc/{pid}/stat") as fh:
                return int(fh.read().rsplit(")", 1)[1].split()[1])

        pid, out = _o.getpid(), {"host": socket.gethostname()}
        for _ in range(8):
            pid = ppid(pid)
            with open(f"/proc/{pid}/cmdline", "rb") as fh:
                if _o.path.basename(fh.read().split(b"\0")[0].decode()) == "java":
                    break
        maps = set()
        with open(f"/proc/{pid}/maps") as fh:
            for line in fh:
                if "libgdalalljni" in line or "/libproj-" in line:
                    maps.add(line.split(None, 5)[-1].strip())
        out["jvm_pid"], out["maps"] = pid, sorted(maps)
        jni = next((m for m in maps if "libgdalalljni" in m), "")
        out["from_node_copy"] = "/geobrix/native/" in jni and "(deleted)" not in jni
        root = jni.split("/geobrix/native/")[0] + "/geobrix/native" if "/geobrix/native/" in jni else None
        if root and _o.path.isdir(root):
            st = _o.stat(root)
            out["copy_root"] = {"path": root, "mode": oct(st.st_mode & 0o777), "uid": st.st_uid, "entries": sorted(_o.listdir(root))}
        yield _j.dumps(out, sort_keys=True)

    def executors(stage):
        rows = spark.sparkContext.parallelize(range(256), 256).mapPartitions(node_report).collect()
        seen = {}
        for r in rows:
            d = json.loads(r)
            seen[(d["host"], d.get("jvm_pid"))] = d
        print(stage, json.dumps(list(seen.values()))[:3000], flush=True)
        return list(seen.values())

    paths = [f.path for f in dbutils.fs.ls(CORPUS) if f.path.endswith(".tif")][:N_TILES]
    src = spark.read.format("binaryFile").load(paths)

    def run_pass(i):
        try:
            r = src.selectExpr("path", "gbx__l1_rst_fromcontent(content, 'GTiff') AS tile").selectExpr(
                "path", "gbx__l1_rst_boundingbox(tile) AS b", "gbx__l1_rst_worldtorastercoordy(tile, 0D, 0D) AS y"
            ).agg(
                F.count("*").alias("n"),
                F.sum(F.col("b").isNull().cast("int")).alias("bbox_null"),
                F.sum(F.col("y").isNull().cast("int")).alias("row_null"),
            ).collect()[0].asDict()
        except Exception as e:  # noqa: BLE001
            r = {"error": f"{type(e).__name__}: {e}"[:2500]}
        r["pass"] = i
        print("pass", i, json.dumps(r), flush=True)
        return r

    RESULT["executors_before"] = executors("before")
    RESULT["passes"] = [run_pass(1)]
    share_proj = os.path.normpath(os.path.join(NATIVE, "..", "share", "proj"))
    if RACE and os.path.isdir(share_proj):
        keep = share_proj + "_t25_keep"
        shutil.copytree(share_proj, keep)
        shutil.rmtree(share_proj)  # probe C: the environment's PROJ data goes away under the live JVMs
        RESULT["deleted"] = share_proj
    for i in range(2, PASSES + 1):
        RESULT["passes"].append(run_pass(i))
    RESULT["executors_after"] = executors("after")
    if RACE and RESULT.get("deleted"):
        shutil.copytree(keep, share_proj)
        shutil.rmtree(keep, ignore_errors=True)
    nodes = RESULT["executors_before"] + RESULT["executors_after"]
    RESULT["verdict"] = (
        "PASS"
        if all(p.get("n") == len(paths) and not p.get("bbox_null") and not p.get("row_null") for p in RESULT["passes"])
        and all(n.get("from_node_copy") for n in nodes)
        else "FAIL"
    )


In [ ]:
if MODE == "standard_budget":
    import shutil
    import tempfile

    import numpy as np
    import rasterio
    from rasterio.transform import from_origin
    from rasterio.windows import Window

    from databricks.labs import gbx
    from databricks.labs.gbx.ds.file_gbx import report_detected_cap
    from databricks.labs.gbx.pyrx.core.budget import session_read_cap

    MIB = 2**20
    gbx.register(spark)
    RESULT["caps"] = {"read_cap_mib": (session_read_cap(spark) or 0) // MIB, "stream_cap_mib": report_detected_cap(spark) // MIB}
    print("caps", json.dumps(RESULT["caps"]), flush=True)
    os.makedirs(SCRATCH, exist_ok=True)

    def tile(name, px, fill):
        """A px x px Float32 GeoTIFF under SCRATCH, written a band of rows at a time (`fill(rows, cols)` gives them)."""
        path = f"{SCRATCH}/budget_{name}_{px}.tif"
        if os.path.exists(path):
            return path
        tmp = tempfile.mkdtemp()
        local = os.path.join(tmp, "b.tif")
        with rasterio.open(local, "w", driver="GTiff", width=px, height=px, count=1, dtype="float32", crs="EPSG:3857",
                           transform=from_origin(400000, 5760000, 10, 10), tiled=True, compress="zstd", nodata=-9999) as ds:
            for r0 in range(0, px, 1024):
                n = min(1024, px - r0)
                yy, xx = np.mgrid[r0:r0 + n, 0:px].astype("float32")
                ds.write(fill(yy, xx).astype("float32")[np.newaxis], window=Window(0, r0, px, n))
        shutil.copy(local, path)
        shutil.rmtree(tmp, ignore_errors=True)
        return path

    def surface(yy, xx):
        return 100.0 + 0.05 * xx + 0.03 * yy + 10.0 * np.sin(xx / 97.0) * np.cos(yy / 71.0)

    def flat(yy, xx):
        return np.zeros_like(xx)

    def noise(yy, xx):
        return np.random.default_rng(int(yy[0, 0])).random(xx.shape, dtype="float32")

    def run(path, opts):
        try:
            return spark.read.format("raster_gbx").options(**opts).load(path).selectExpr(
                "gbx_rst_summary(tile) AS s",
                "gbx_rst_slope(tile) AS t",
            ).selectExpr(
                "s IS NOT NULL AS summary_ok",
                "t.raster IS NOT NULL AS slope_ok",
                "t.metadata['last_error'] AS slope_error",
            ).collect()[0].asDict()
        except Exception as e:  # noqa: BLE001
            return {"error": f"{type(e).__name__}: {e}"[:2000]}

    FORMS = (("virtual", {}), ("materialized", {"virtualTiles": "false"}))
    RESULT["decoded_mib"] = round(BUDGET_PX * BUDGET_PX * 4 / MIB, 1)
    RESULT["rows"] = {}
    for name, px, fill in (("in", BUDGET_PX, surface), ("in_large", BUDGET_LARGE_PX, flat), ("over", OVER_PX, flat)):
        path = tile(name, px, fill)
        for kind, opts in FORMS:
            r = run(path, opts)
            r["decoded_mib"] = round(px * px * 4 / MIB, 1)
            RESULT["rows"][f"{name}_{kind}"] = r
            print(name, kind, json.dumps(r), flush=True)
    noisy = tile("noise", NOISE_PX, noise)
    try:
        spark.read.format("raster_gbx").option("virtualTiles", "false").load(noisy).selectExpr("tile.raster IS NOT NULL AS ok").collect()
        RESULT["rows"]["noise_materialized"] = {"refused": False}
    except Exception as e:  # noqa: BLE001
        RESULT["rows"]["noise_materialized"] = {"refused": "over this compute's read cap" in str(e), "error": str(e)[-600:]}
    RESULT["rows"]["noise_materialized"]["file_mib"] = round(os.path.getsize(noisy) / MIB, 1)
    print("noise materialized", json.dumps(RESULT["rows"]["noise_materialized"]), flush=True)

    rows = RESULT["rows"]

    def ok(r):
        return bool(r.get("summary_ok") and r.get("slope_ok"))

    def refused(r):
        return r.get("slope_ok") is False and "read cap" in (r.get("slope_error") or "")

    RESULT["verdict"] = (
        "PASS"
        if RESULT["caps"] == {"read_cap_mib": 1536, "stream_cap_mib": 256}
        and all(ok(rows[f"{n}_{k}"]) for n in ("in", "in_large") for k, _ in FORMS)
        and all(refused(rows[f"over_{k}"]) for k, _ in FORMS)
        and rows["noise_materialized"]["refused"]
        else "FAIL"
    )


In [ ]:
HYPOT_VECTORS = [
"bdae704b1253716b 3df2f9e640c07c42 3df2ffffac11ce55",
"3dee7f22b16d8d6b bde11f82a0b8da1c 3df17cc9c1d57bc6",
"3e1056534d6ffd75 be25a51c5648e202 3e272292a623faf4",
"bdc6d08a3d0476f0 bdbedf2db62ffc7f 3dcb8bbc23252d90",
"bdf8354ff2fe7689 3df74a8fa69e71db 3e00cbf72e8d6ced",
"3e10113d19d6adaa be04164b66722254 3e12f2b989a76615",
"be02f37993227cbe 3df3230a354e32aa 3e053aca93a9bc8e",
"3e188fc54ac8587d 3e1a2db3b894c2d5 3e21f2cda660be07",
"3dfdf2be3ef7d801 3df1f7e4cb9c4292 3e017666e3636d70",
"3e06753d68681778 bdf881afed1accea 3e09955107c307d0",
"bdc8fb8ef3172fb5 bdfd751c5aea713a 3dfd9f5e1844388b",
"3e18b63828dd9556 be025f898c0edd42 3e1a5d365c7e949c",
"be01c0d9db644723 be1933976da49abb 3e1ab81946f7db03",
"3e077dcb7a7368d3 be1598d3ae9521fe 3e189598be02bed8",
"be2bc8988741af4d 3e1aa8f7a24264c9 3e2ed0db60eb04fb",
"3dedf96e0336ad6a be1fa302fc767f96 3e1fdb9cc6cd7b89",
"be1496954bd156fa be01718408873c18 3e165c01c39401c5",
"3e177f58a94da69a 3e082306ac3d8cd9 3e1a6a5b9544c1bf",
"3e013d713bd5405f bdfa047607d00a36 3e0598ea694c007d",
"bddf746bad804e79 be15e8d325655183 3e15ff5bddd7da2c",
"be10e16e27fc38df 3e0c368d299fb9e0 3e15ffb3d1469059",
"bdf3bc589f74d4b2 bdccf9f99cb86078 3df410bab8b44d3d",
"3e1055b328ffc77d 3e2158bbf02f5bee 3e232c564be41690",
"bdf682ad23c1ed19 3dbc0cbf9c71e5e9 3df6942043eddcef",
"3e0809893b6fffc9 bdf821cef78be443 3e0ae54c6a8388ef",
"bdfe460267aface0 be1ef1dbb0776519 3e1fdb59e3c3a2ba",
"3da1930d1a226cf1 be0ccda28ad87c0d 3e0ccdf85340486a",
"3e09e92036ae4510 bdf9862427aa5616 3e0ce20f1bb3a718",
"bdf90a32b62f353e 3e1d5fd8cceba860 3e1e08b5c4f36faa",
"be12470fbfc580b9 be12eeb7e90204ae 3e1a50c969953f0f",
"3df7430507367619 be1f8a7b14ae2d9f 3e20094a4bc5e616",
"3deb56981ad8e6fd bdf8173abfe4b1d0 3dfbb2d335552b50",
"3e20d4e92c3ab824 3e0b07b164d184c7 3e2223345286443e",
"bddcd06927d56392 3df97a9b0c837976 3dfa7a480cadd89c",
"3e078007dcfc564e be11f0e9c732cd5f 3e157243cc939849",
"be0d2d40dde545c0 3e13edaf2f55c5fb 3e18b28c69c6a4ae",
"3df94627d2295dc1 3e14584b13475314 3e154db104df6a7e",
"3e05018513be7ec2 3dff9fb96c6e4ea6 3e0a4abe53a81cab",
"bde868a585374dda 3e20432729973a7a 3e20556e50bb3018",
"3de4617dd9fcd4c9 be156149696920c5 3e158801a7204692",
"bdfe331985995545 3e05eae4c4b0e6ff 3e0a9d8f6ac0dcd8",
"3e12bddd8cf28065 3e0db63551faea9e 3e17ea586f6ba946",
"3e20046a00b8a627 bdd28cc360ef65d5 3e2007193b39fc5b",
"be026b925233bb7d 3dd4500d0a951239 3e029828f782accc",
"3e108f70a4ce3e19 3e105d5ea4abaef8 3e17482c755819d7",
"3e135dbaf9a5e37c be035616cbe9062b 3e15a53416441e24",
"be1051d473262a5c bde06f415e2af731 3e1072ccebb0bb04",
"bdd0f5e4e6258f99 be24041dd63c0c76 3e2405e9a43a18ff",
"3e1983a64c8a5333 3dfb74db9f78e8b4 3e1a6be49f5b300a",
"3de7cdf907703afc 3df6a780e1bca1b0 3df99736076cc232",
"3df4f650241ecbb1 be281b408ca6ba31 3e283f99e0129149",
"be1224fd7347347c 3e070226869b890c 3e157bf20d533a9a",
"3dc6c5c4ffc19fc9 3e1069631e15c9b2 3e106d55cd99041f",
"3e218907bc88f678 be0bd5cd50043b78 3e22dd9788b3d184",
"3df5f6f279756063 3e15fcb08a56dfd3 3e16a9928ac8bb73",
"3df1f53f3d860341 bdbf452c5f56939b 3df210646abc5e0b",
"3e11b23f578be540 3e041002c68f8ea6 3e1457798b62619f",
"be0f6d5ee4bb8129 3e03c7283f93fab8 3e1290fcbb01b615",
"bddeefdd9cfbd090 be1115a39c90089f 3e11318f839d34df",
"bdef14024afe3617 bdb27f7f3ff46b47 3def29ffbea82e02",
"3e267aa7f4d38032 bdf7ca83cd473747 3e26accbfc83b9da",
"be014539556868de 3e08395103d91584 3e0dc001886211d6",
"bde4e1545f7e549e 3df6be1d63655fca 3df906425f5ec70e",
"be0cfcfa52a105a8 be1958cc0a750592 3e1d32c59b12afb6",
"bdd57f2cc847523b 3e07c486f98f7f65 3e07eb4a1a86cdd9",
"3e1e5b8a06f2b440 3e0934b3ccf4776c 3e206f51e4eb6b7e",
"3e11b1e3c338176a bddbbdcecb8614b5 3e11c7957372b77c",
"be20c0389ca0dd68 be0c788c931dc418 3e22334b615bf929",
"3dcbf9d0aada572d 3e1d456f73c3961c 3e1d48c6decc7ed4",
"be1b45e736dc0333 be1906a4c3543225 3e2281eef7f9cfa0",
"be10c15b6657db86 3e02cdf4a29dda28 3e13368c9603a402",
"3df3bd36c96b14c8 be0d254c05698e88 3e0ec57af2547f29",
"3df4e964362100e8 3e125ab4bab68ca2 3e131597a5bc513a",
"be05b05da8b265f7 bde77d503da15085 3e06784883fe69b1",
"be10bb7ac0b1e51b be026967902973db 3e1318fdf02d0430",
"be0a200cf3670d1d 3e1820bc0bb134ad 3e1b6fdb2b3c7af9",
"bdec2eb669a5af72 3de2da77d5d43c9c 3df0f420f831aff1",
"3e0fe007386188fe 3e2d78400f6e71ea 3e2e87330684d22f",
"3dff9a5679427291 3e155f1286e8379c 3e16c8f70a7ae082",
"bde68fd609a7c646 be1f9edc17360bcb 3e1fbefe19f02074",
"3e059352986ee6bf be196cae2c5408d5 3e1b9e56c7877f13",
"3df8f57d8ae9b81e be0aa43cf3762e01 3e0d6b67651849a8",
"3e00d7d5a18d942e be19f8af3102cb84 3e1b4d7da9d01287",
"bdfa3be5a5861493 3e22c1f445692b8e 3e230ac845e1bee7",
"3e0eab478a6f4c99 bdfc09ce7e3f80dd 3e10dc5475e2d7cb",
"3dfe848684b537f8 3e12c8c5b8e19c1c 3e144647db77fdb5",
"3e009fb18d44aaa2 be19775d06a2031a 3e1ac9d473d71ff0",
"3e0c758028e555a5 bddc02198b88b1da 3e0cac6c379fc0ba",
"bdf7f4c719c7f65d be094d4a044ed85f 3e0bfe735ed3ef0e",
"3df9b7c02878ade3 3d905d26901d54a3 3df9b81373f07aed",
"3df15b302f5e3701 be0424a5031478f6 3e05eed7d947948f",
"3de08f8dcd225997 be00184daae1170d 3e009e70c1c6170f",
"be232062e756715f be16035e943ed2be 3e26113e34f48b35",
"bdfc7a299c86afa9 3e0b9b1ac3c4e4d7 3e0f0fc41f7f3381",
"3e21b262ab7fee28 3df70a1e5f08d908 3e21edfbdc575bca",
"be0664dc52b051b6 bddaed10bcd98e47 3e06a53fe51b2485",
"bde1240d55d7373d 3df89288dbb56d48 3dfa062ccd80d4b5",
"be0d4649c7d4018a 3e1a17340e73c026 3e1dea83be723b0c",
"be05063e9115f45b bdf0bc6cd6cdafa3 3e06a0e50df55a77",
"be0303fb488721af 3df565ee2ed83d53 3e05d19f465541ae",
"bf2de9e6e1d4c111 3f5dc9428d06a5ef 3f5e051bcab4b1c7",
"bf4f0695ce63b4ed bf2eee29b985f4fe 3f4ff98cfd4c8c13",
"3f45776ce36d589c bf402d48df672544 3f4ae1210fd764fa",
"3f31ee5555dfdeb2 3f39fa2283862eec 3f3f908f69e7ae99",
"bf45ec71b6be6575 3f35b356c8ea76c2 3f48762fb3aca37a",
"bf6418072f284247 3f486409edfd9258 3f64ffaa48661766",
"bf65b64b549bcd5e 3f39d4d9ee0d26a2 3f65f36c404e3fe9",
"bf5163b639ed16c3 bf44f9d8027fd3e0 3f544eb4ba2b2ab8",
"bf43e7f3f76beb3e bec44e4a81e57646 3f43e7fe52c0cfb1",
"3f25ae3ffe01ee96 bf46750e11625cff 3f471a21e799cc45",
"3ef4138056ca5e9c 3f4d4b79c6189535 3f4d4d31fecf6114",
"3f49271e1e53caee 3f447792883504bc 3f5036c37d762e15",
"bf3da42c2b9f4be6 3f4732c527bdbf23 3f4b874655e34dd2",
"bf12fb31613aad75 3f537b3e7fd77f9d 3f53847b860c1b5e",
"3f5d85f174db6195 bf137d5b9c290dca 3f5d8c5f9cc8940f",
"bf5ef1b241a24593 3eeb97f3f9aeeeb1 3f5ef1e377a64568",
"3f4b586e64c42760 bf463e9d35d4c873 3f51a00ed49c2943",
"bf4ea92e38ec97fe 3f2e279734620648 3f4f92f42524b472",
"bf59230b4aeaec30 bf4932a9418c6be4 3f5c1e17efbc3a14",
"3f35e7f970efa636 bf41eca1dbd30dd1 3f450189237fd46c",
"bf54ce6c9b5d05b2 bf3f628dafe98475 3f563c937f3026d7",
"bf3313d91c330fb5 bf1e5b05c1bbb474 3f348811e99c4992",
"3f088fa52bea6310 bf471f6284494b4e 3f472c6a40d886c1",
"3f62af1d04b3fa26 3f3b9c16bab2183a 3f630007159c40a8",
"bf5de0b40a45d436 3f4398f08e064b2f 3f5f718a0efa6a97",
"bf50c2e9ef191afa 3f4dc02aa5955923 3f56690609e7ddae",
"3f5a4c7ac7c1347d bf55631261306afe 3f60f2db30a16b04",
"3f0a1576100eed1c bf532b87c3680e52 3f532ff6f76b71c9",
"3f501834d5ab9397 bf34a58a40a5b7ab 3f50e6e938dc2229",
"bf4bdb17a18b0a3e 3f513b8d28a7a739 3f56284af246c4e0",
"bf35296f23805813 bf294d0481b5013b 3f38a79e69a8def8",
"3f604bf88040aa44 3f4ca5d3c75bde1a 3f61cd1262144ac1",
"3f36f0a24d1a211b 3f3f696125470cff 3f4372bec364e6ed",
"3f5452bb3f208b2a 3f2791effa0b7374 3f54891e5a52312e",
"3f405699557cf5fd bf45e0d59524baa1 3f4b4e3ddff05c48",
"bf47f38bd805fedd bf5a2f1c766eb3a5 3f5ccaef0a90b284",
"bf5720f93d17a3bf 3f30b440af301292 3f5780b64b16acf7",
"bf475c24a7c85377 3f3e90152b3d566f 3f4bea0f79e03228",
"bf41fb077465f0f2 3f44b58255d55786 3f4b6cf5b34bbb74",
"3f1db6304215500a bf5144f3c4406d3d 3f515e70221238ff",
"bebb940137c4f3f7 bf373c83fda5e57e 3f373c945b36d5fb",
"bf40393ff7c6d5f6 3f4d0963211970fa 3f50a17c47d1f09b",
"3f3b06b3dfd2616c bf608144e71c3d59 3f60d8de9bb97a54",
"3f365012b23c3808 bf43a5893b1f6ad1 3f4697df4808b4a1",
"3f3ead97fbc7b565 bf54e526c422075a 3f564219d7902eb2",
"3f511f86ebabffe3 3f3ab9f30972c908 3f52616e354b0c40",
"3f47b7df6d5ab751 3f54c16bd5599af6 3f57e79679ecbdbe",
"3f5ff86b536760a1 3f44e1e8f56a43dd 3f60d0ebfaeee889",
"3f503e79a41f0d49 bf5a1e306f10eeb1 3f5ec1e22e0ae584",
"3f1b3464827d41fa 3f48605461db660c 3f489cc26a3c1ba7",
"3f420e84442726dc bf3ec999f83a8111 3f47ba56492ebde3",
"bf3e885576ac227d 3f377d501e98ac92 3f4342e5ce1972f6",
"3f5cecc641c2c808 bf455fadf2cba104 3f5ed60456e442a7",
"bf489eeb2d279b0d bf315a2ae1534ac0 3f4a1ad04348c9db",
"3f3751019b43cd8f 3f3a1400fdfe8f13 3f417dab2815b6d2",
"bf1cf06ca8c45aae 3f40a3556720e6b6 3f4106d75bfe60c7",
"3f2e7896c7180b52 bf5a6a8d893ecbb4 3f5ab07ce27ed6b1",
"3f587dbaa628333b bf26fa28fbad01a2 3f58a8b20ebff3a5",
"3f5d6de1b948524c bf4a189c992dc6be 3f601896cd648962",
"3f55f7ab49686311 bf53672483283f6c 3f5d4f35c0c19165",
"bf51fd778589be55 3f511f1b993ef0bb 3f58d5d39cecfcfd",
"bf443a3b4a9e4619 bec39c20285253dc 3f443a44cc087647",
"bf3dd13bde106a68 bf4550f6559eb372 3f4a03344c0857ff",
"bf5c1775bbff8037 3f43c51a79433e7a 3f5dc7b2f5e8c673",
"3f11a55842c75284 3f3f1a169536ffd7 3f3f69c80d6d4319",
"3f540c8c8d3b1351 3f13da37acb875ef 3f54165e5636cdb1",
"bf45c213d4e7b8fe bf154d58f5468da1 3f45eba2463c7fb7",
"3f519051d3875026 3f3854c4d5d76b13 3f529655616c2331",
"3f51bd03fa838db2 3f38270d4d40e849 3f52bce5382d0b39",
"3f3c9798fccb6f55 bf06a3ed0668fa0b 3f3cbb5d781a3d03",
"bf52b65a55a6f8d5 bf520d30f1178423 3f5a000aa01ccf4d",
"3f2aebb6b505c987 3efc8786892fe6a4 3f2b27eb469bf069",
"bf16475166150fa6 3f426aff310fe927 3f42a096d3bf4271",
"bf5247a7365773d0 3f46e3fcb382896a 3f55913bb314e64c",
"bf5666ef0828bc89 bf47f97ff5804831 3f59685bffb85735",
"3f3061a64561b002 bf56d4ae5283f144 3f5731f7ce5208d6",
"3f47201b722ea503 3f509f76e52b3799 3f543fb6e2678478",
"3f43d5d2e8b2dbcd bf375d5c1b1357af 3f470515fd7c30bc",
"bf47cf4d16eab8d1 3f26c1c5fa16b8f9 3f487ae419f53116",
"3f3bc75e6bb1fb00 3f534f1320721641 3f54851118299f4b",
"bf4a2c5002200b70 bf446922febc4c0e 3f50986b27be7031",
"bf546a1869d20923 bf415942f71f7d3f 3f562e50ecbffbce",
"3f6ac567f81b8e57 bf562e0c0d87f420 3f6cfa2b3eda8d41",
"bf51d8d3abd09639 bf51947b77e43150 3f590d26eec78229",
"3f5d6b15440235e1 3f53532effce3703 3f6199535f4b4a27",
"bf316bb9bae290af 3f40194964e1254f 3f424ddc60d32940",
"3f3f621441e374d7 bee7e439dd32a849 3f3f645a32fad098",
"3f1c782b047d8ea8 3f59bb2be23b0d51 3f59cae6f4ff80fd",
"3f549b1656c67a2b bf53df718cc6c60e 3f5ca09df583b70b",
"bf48fb4977258a92 bf6220d6403d6d13 3f632c85e4d13ba6",
"3f513a7ba3e0833f bf49cb4d39cc6b22 3f5585610db3aee8",
"bf2e3fec0df5c56e bf2de01e81a465ad 3f35420387502c7b",
"bf5b45f78dede950 3f2eef6b1d663389 3f5b8bcba413daac",
"3f354ae40cc7070d bf55b7858dd9d063 3f565c1903e10ac0",
"3f52323b4abf37d5 bf23e5ef49f6c2d4 3f525d8c8394bb45",
"bf42ece9b0f61b68 bf367e7d93cd9fb5 3f4603e356055912",
"3effc92e6a8248ff 3f5436d3ab82e3c6 3f54386376ddce5b",
"3f4bfd6de53066fd 3f3dd10095cff9cf 3f4fb6707a343a22",
"3f5f0a84fedc295f 3f2f6e8d3e565913 3f5f49ebee8c459b",
"3f55bf9daae712af 3f4e1a645ebca784 3f5a72ed934f1570",
"3fd5b036a28d1745 bfe0fbcdd7ee2099 3fe426812756104c",
"3fcef336f4852b60 bff0c5e75a39d999 3ff136a4adaaff86",
"400a30cdcbcb7591 3fe2c6465cf55391 400a9b9ea2fce4f5",
"bff5ba8aec34cda6 3fb3590e8ed966bb 3ff5c32664dcab73",
"bfe980bbdfa663c4 3fea740d0cb6e584 3ff25f5309e4045d",
"bfc50d078902afa8 3fda8e18f9619edd 3fdc90a0f8e1a1a6",
"3fc71e651282d50c 3fa2b212050b8e9b 3fc7962213eb17ff",
"3fd918a70eba3879 bff02ec4280ee688 3ff15b3a48945689",
"3febb1dd3f5b6d03 bffb1aff45f92b38 3ffe700fcadf67a3",
"bfbc360b424b79a4 3fec907d6ccc5a4f 3fecc800ea118389",
"3fa7691d376c36d1 bfd5a9a6a41e519b 3fd5dc0571ad0f96",
"bff1f055f9825a11 bfefd1e7072d6d67 3ff7fa47a79df8e6",
"3fb64108a6aaae64 c001230d85ee2607 400126a9ea35f4e4",
"3fe433be98b6dfec 4003cbce19ce072c 40046e230d07576a",
"c00136db7dd129e5 bfd7a2870391be36 400177496cfed0b2",
"bf819b3a2a028e6f 3fdb1c269291b3e1 3fdb1d946ecf31d3",
"bffecf73703e4990 3ffdd4c25b8ae02c 4005715a4c46322c",
"bfdca44128c51792 3fdbf64fa5c7142b 3fe4038ec65d2103",
"bff7f041cba67b01 bfed467ba500ceb4 3ffc0f2136849906",
"3fff9a2cab07c34a 3fe389b34ea95332 400089f2066fbb0a",
"bfd98d2874e7da2a bfdd2869fbe4e0bd 3fe36276f88b16d6",
"bfc9dd7745bccf6f bfd0a381274a4625 3fd512d7d0ddbc56",
"3fe8717510822dc3 3ffc663bfbe059c9 3ffeeae00ee6f39c",
"bfe5978b29a62589 bff4564c0d120704 3ff70666ab9099c7",
"3fadf9c0a8a296f9 bfe4bceeae8d9b0a 3fe4d28d64b4f41a",
"bfd65d67c6dd52a4 bff9c92efeeb87fc 3ffa62953e763c64",
"bfe572801b8e9205 3fe31d5064ca60f8 3fecba992e95fc7a",
"bff153af65603a4f bff1e30a1883740d 3ff8e72dc654d501",
"bfced7c88796396c bfe7a3aa96117d28 3fe8dd76b34ee4d1",
"3ff1113e8bfe30e6 3ff9102ef8637049 3ffe52957039ffc1",
"bfe4ae38dba38cbc bfeccea19750d81b 3ff1bb1b6b309c39",
"bfb6bb04c7c3882b 3fefcb430b698241 3fefebb2e73d96e8",
"c003767ee24338d8 3fe10ddde6cec33f 4003eca4b1281115",
"3ff01f5ae217a176 3ff10cec452760ce 3ff7774be94fc348",
"bff84b79dafa48b1 bfed64e35e4d5402 3ffc64f74dad692d",
"bfd80e38b1c85fd8 bfe1d6f4427702a2 3fe583fe59a41c6a",
"3fe6582332c0941d bfd41759eed23605 3fe87fa4411ba228",
"3fc4f6dafd007aa8 3fed49bc93055fe3 3fedc0d7056fc033",
"3fe3ecd0d1ef0228 3fd0709b35e98a96 3fe58dd3a901f780",
"3fb04658d179537c 3fe308f23ebc1331 3fe324b26e8e3fe4",
"bfefef18696a6cd8 bfd96b97dd470796 3ff12f6a003cb685",
"3fbce43c85c30153 3ff423efb52205ae 3ff4389e129de077",
"3ff7526457f803d2 3fe3545662801812 3ff93ebf146b35c3",
"bff38b00f814f17e 3fee9470090ed867 3ff8d04186e6c8ed",
"bfd71a003ef7e404 bfe4bf5f1069c638 3fe7bf05d1e087f0",
"bffde5f97f2b6325 3ffada19c14b22f8 400417d872ee4539",
"bff5ea73f4b4a474 bfeb210c36a55ae9 3ff9c626cdda1024",
"3fcd67e36f298381 bfdd2e082441491d 3fe0565d01513928",
"bfe0d4f19f4ebfe1 bf803d9d69cb7b03 3fe0d56efa49f466",
"bfe8b75c7f2c1446 3fec67515de5836d 3ff2d36db249cf32",
"3fe8cadc0bff0770 3fcaa0b90ea2d09f 3fe9ababbee4f729",
"3fe2b74189d2068d bfdb4065ea41fd93 3fe72685b6976fb1",
"bfef7485a10609e9 bff0991b360e621a 3ff6ddb0895e1f41",
"bfc598ad72a0391e bfd446a18f6ba2ab 3fd6f8d5c0bbe0c0",
"3fd1876c5697ec57 bfe1d0ec75c07992 3fe3daf052d0e39b",
"bff8866e3f438344 bfe271c76ecd0150 3ffa33a4ea96f05e",
"3fed7a2831da9b98 3ff757ff0815dee6 3ffb9b6fa063431a",
"bff6c83fa2719834 bfe3e9873241f50c 3ff8dcd9b8fe6924",
"c0024b0df788dc25 bfe0d0bedf308061 4002c51eb1216206",
"3ff98927e053c11e bfe4c0c5ed6ba89d 3ffb904260d0c3ef",
"3f7bfc37ba3e32ff 3fdbe92f4da7e51a 3fdbea0fc48a58f0",
"bfec72f7b8040922 bff9f92a9520dd79 3ffd9d0454730ddf",
"3fbe9babcf67c07c bfd9b38d1b3ed593 3fdad0f90f954115",
"bfe033d5ca565238 bfdbfd239191f552 3fe568d0ce606d98",
"bff875edfbd4a914 3ff05c7d9a7fb5ef 3ffd6d9b161f867b",
"3fa1b253a9991131 bff48b1bd2dd01a2 3ff48d038d663d1e",
"bfa5f3cab8e70db1 bff8166aa6b70cf3 3ff818eab96e0334",
"3fcc5bf6d2d8bafb bf70f1cfe0f7d244 3fcc5d3ac644b635",
"bfcaeab247d94ef1 3fcc55445a9adb3b 3fd38a49f928549e",
"3ffb53470ab67754 bfae29c06f17b913 3ffb57702f1dbf89",
"3fd72447852bc1b4 bfeb4dfb031a75a7 3feda7b668667564",
"3fe201559e289842 3fe567435c60f6cb 3febf8300fa24b09",
"3ff3a408010f6668 3fce8cc7daa13a52 3ff4022fc44abf6e",
"3fc1798a4d456b24 bfe728ee4aae14ef 3fe7917d8812fb35",
"bff1beb665a1de77 c002955268fedc52 400497bbb3564c3f",
"3fcb958b75a05c9c bff14ce1b6e5a704 3ff1a3fcb7a9b65f",
"bfec9aa4f091cca1 40022553daadd533 40038108b1359ff1",
"bfb468b8bd609a3a bfc0635c93bb616e 3fc34e370494fdfd",
"bff21e93ac31492f 3ffc4b6935d54556 4000ccae1be8aaa9",
"3ff23de49d333098 bfef76ef89003a0a 3ff816b78aba6ed7",
"bff6b69ec7277f9e bfe6ed2957db9b8b 3ff9712f3c78e4da",
"3ff0dbb1138a865f bfd851bebc1d42a6 3ff1ebc6a8b2f196",
"bfe71d9b14ad4ec7 3fdba8b76c3b3aef 3feaefcf58475a76",
"bfddc58e3d8d0a04 bfe620f5edbf15cb 3feaab6e333913ee",
"bff5c8295c91fa32 bfe4e7b7222926ef 3ff828f70ae942c6",
"3fec706b4e599df1 bfc4a4ff43d295d2 3fece75687529e24",
"bf93ac7d74192ef6 3fc66ef03ddf1506 3fc69157bf042421",
"3fea8affa8b366d4 3fe43b689f96af11 3ff0aff39ac7a052",
"bfb4d78f1ea3d1a1 bfda22c31db2cadc 3fdaa66e40dec237",
"bfe9a7d480b3ff5f bff4037651d3ac09 3ff7c58db9a8dbc0",
"3fef9bcd52a46fea bfe16f087748c39b 3ff20c7eea7d0fcc",
"3fd3795e8ad62c96 c0032adcabba1ce7 400352466a23ad19",
"3fc83078721d8963 bfe1998e6b08f468 3fe29c1b227ffe88",
"bfd16d13eb43188b 3fe96b4dfa0f7234 3feadef9f83f5965",
"3fd7cdcdf31027ea 3fe5fd0c47284f60 3fe900c3df7d2ce7",
"bfe1ddf5af1e0c1e 3fee8ea21884b609 3ff1b2dba6bf64d5",
"bfeb38d7696ecd18 bff917d0ba3e9a83 3ffc8bfb3972f061",
"bff5f44d0f0147b8 bfd64e26d03c4a84 3ff6a6c33b004c95",
"bff5dc3a619e5e9a bfe86be464b120a2 3ff90a181bbd121d",
"3fe2428837e36e8d bfefb4bc77084289 3ff24b491e5db571",
"405a2d1638df3c1f c05fc7b9db76db16 406496123a74c490",
"c06fbcf1b3eb4938 c05b7f499a0eefe8 40714b432cfcc684",
"c0308d4d493a96e0 405accc1be91af52 405b1e0e9374c738",
"c05d651d3f0b5d1a 4010bd425124f599 405d69e10109a036",
"c04a677f65b114da 403d9f86bde64335 404e465f668389a2",
"40279313395fd51a c049d8ef83a39f02 404a82c49c1853ad",
"4065502a1638addb c044526135f47a7d 4065e9090a58bc07",
"405a51a978823f54 402cffa368f46e3c 405a91438846a976",
"401b4b1b16462cb0 40633437d3c0691e 4063391081a0c72d",
"c044ca77193e2d52 404f800ffaae9ecb 4052df172624df5a",
"404552a5d5e44623 c05c16e973200b8c 405e0b73884bf8d9",
"406e14148551d58d c0562ff9efe21740 4070078ef55bc996",
"c04c45dde1433cd9 c061866354fb42c5 4062e58baa1fccda",
"4062491b7038e029 c0423782dc330ff8 4062d81c1c94bd23",
"403d726244dce836 403c417e96bda784 404467c28ec467d1",
"40568d62b8446286 c05580eed067b6b6 405f293bfc94d540",
"4031164a83363687 c05a2eb325f849d1 405a875306705743",
"c057b5005989ddfe 40570e1b66747db1 406088d65685d27b",
"40312d636205d06c 4011e05080f88779 4031bfc7beae8bf0",
"4040a8682f866667 3ff739842656d434 4040ac73dcf7b775",
"c02a9a41ef566fdc c04f807056136536 40501919e5355a86",
"c06e9c90643c1e22 404833f831cb5651 406f342f93dcab2e",
"4034823087aa3cfb c053be41a03cffcf 405465e795906f60",
"c056400fe10fc681 406651e2c90344e6 4068f0568b4c313c",
"c0318fece3ae95d1 c0533fe69279614b 4053be7514c58aaa",
"c0465b2386f215a4 c05d826f8da97f5b 405f8e3f4d0ff35d",
"c064ffef8f02714a 402bb19104468d5c 4065122a5cdc4ac6",
"405109714ea195f7 c05a1ed705a7513c 405f2f7bcb02aeb1",
"40396281d976d438 40555cef528a6575 40564924b2d67084",
"c0649abbc24dc3a6 404654019452e1c9 406558dffca1318e",
"4042ef1267a04bc7 40353378b4dd97d0 4045b30a33584046",
"c067e0fada7d36bc c05d7f7429fdead6 406c110755a28d02",
"405d369beb877f0c 4054d7d6e787c849 4061f17ea87511ff",
"c03a257333ed5180 c058ca4757f4d49c 4059a32f51935df9",
"c0508f4613fdd275 403992645ed3785f 4051c0372e14ee40",
"c057b4ea3ff32eee 4045f99e61352347 405a211151de421a",
"4055125027715c64 c02bfe371e00ac5f 40555c2ef5d4e599",
"404500af36a37ab4 c054b2bb65e6b0c9 405735b2098a4a8e",
"c05287b78fb8c983 c054feade4cdccbd 405c00af12eca721",
"4027668e2752ab1e 402056b44f4faa35 402c8a4864ee1739",
"c00dceee6f53f544 c0550314924f2414 4055085d178d64ff",
"c012e07b6da54a6c c03954633d2b69ff 4039c3f8ac62f833",
"40565537a16123a1 c036639831eab326 4057060ba50b4350",
"404ec47733caa550 40522d1ff55bdf7b 4057d000ae911863",
"4059dc5af5ea9388 406742225f79ead6 406a9c6a1b3c7504",
"401130d81adfe6fe 40369dd58e2cf6e5 40370570d7f09262",
"405fde108b3954d6 c04b7b40b2ed722e 40615a1043945b21",
"c069e5bbd91afe72 402387917822f19f 4069ed17ec047563",
"c04b939968bf3011 c035fc9c949a7ff5 404dafe2ad033272",
"40409f9b49b0de4d 405ec2277e47d15a 405fdc955809aad4",
"c05755af27a33ccc c008a2b575008f90 405758ef3742c0ac",
"c041b1995942b627 c03e5f1267b9cb71 4047511aa233329a",
"c06113e941418b64 40583bd817193fc2 4064f08d8bb43a5f",
"404417c478050d7d 405dcf36bf8762c9 405f74f3507082c4",
"405248c419d8c2b1 c0357857b45ace60 40530e496a5a76aa",
"c05ada1e2a761069 4032f2e4b47d49ae 405b444592736694",
"401edadb46b9013b 406357d035b8052b 40635df635a8f39f",
"c04ee2fc877eadcc c05c1ab2dbc7504b 406008afaf7b23e6",
"405aaf65d3c468fa 4016900323dfe084 405ab8edf633c19d",
"c05819c654608d60 40250012ddfeb63c 40583e43a90b9b91",
"4057fe2e37d7b393 c04c9b91cf9ad546 405beee0a6881b30",
"4045d4606fc14323 c063db7c5a22b2ec 406497f917c39636",
"404e562b80098768 40335349738f7d3e 404fd69a05a0a016",
"c0384b9f1815794a 404d76013e4deebd 404fddfcccc4d051",
"406355aa96d7622e 404d13cbf600b8f2 4064a7f11cd5eb48",
"c0429ee86ff4391a 400ec6beef92d8d3 4042b84627dd3a27",
"405db9031fe387ce 404a48219f9154ee 40603fbf6af8f8d3",
"40640ca6dac67e80 4045441d7703c4bd 4064be09f82a36c5",
"c01ad8d371ebc0a9 c04fc147fa39bbd6 404fee8cd863fc3b",
"c05be79e47ae92d8 c0432aff62b513ec 405d813508e8256d",
"40471b8086db2643 c05791468777b2ff 405a3f47502d1835",
"4026bdc839effaaa 403315eb3a5aa51e 4036374f3e94f0cb",
"c052b7c06144fe02 404b15ceb7ee57d7 40571a6a385f7c35",
"c03f53d67dbc2190 3fe1f4f9ad63a609 403f551fd69c2336",
"c06933af6b65a324 c0683d6aba82ad02 40717bceefaa301c",
"40535ba38bc3c1cb 40423d1f319f2ed7 4055660119fad84e",
"40136b0a32c82afc c057f5be3c9ee316 4057fd9b5098ece8",
"c054377ad5f6a762 4062a3549c3839f0 406533e3e8232f9a",
"402cee07ba9d2891 403196cfbcb2691f 4036c5e428cf5d8c",
"405ebde0979c0a28 40493b7282dd56e8 40609d69b128dc6a",
"405836bbbee96495 c04c9cb7e39ef8ec 405c1fcdb0d6b20f",
"405e4e2d26c553e8 4056833cf81e7180 4062e04af4caff8d",
"c066c3d12b9cc5b4 c0664bc3791e178c 406fdd543ab820ed",
"c045e4858643905f 4054926c64a2eee8 40574d8ed29fe101",
"c050361e7526a7ba c05fe7613adea435 4061e4a78959ec45",
"406912d3dfe79f15 4045a569ef075052 4069a69fc6a2f285",
"406b2229b52c4e19 40300dcc4cfd7d4a 406b352251a0bbde",
"c050b502eb28f18d c0605daf6d71b192 40625fe63ccf63b4",
"40744b5b81d294f3 c060f0da5f033c20 4075fdb9cdcaca4e",
"405b6e4cf2301f42 40425d35e27c27f2 405ced4594292de6",
"403888e7ae829b8b 404e2d8a32e5322e 405049b9751d1044",
"c05006701c82c2e5 c0508ae75afb80f1 4057082288178658",
"c052f2ee04d081de c059dae9d2b62142 4060071828e355b3",
"c02160c39bd14ff9 c04f8ca591a008dc 404fd8dd301b329e",
"4022e202452268c2 40573f23b78495c2 40575dbcbedfc8b2",
"40696b67e410aac5 40432f6024af9e47 4069de3ae63a9098",
"c05578a5c93fb059 c0485b45d650041a 4058af3d63368790",
"4069cd2d2975bdc6 405a4dc25f623c13 406cf5c9683e35f4",
"c04ed49e57186904 c04ecd14f1bd5716 4055ca4294dbc50c",
"c041331eec4dbeb5 405e027b9d626a19 405f37b57ad60745",
"c0f5ceec178f60a5 410040bec334137b 410392650e0b14e0",
"40e39c4e32ecb0ce 40ab7466250acc53 40e3af7cad5d8777",
"40ca8bc10eb29306 40fcd27923c6b63c 40fd03360b1d4098",
"40f1302163d9abee c103fdde569e3157 4105c2b706ce0453",
"c0dd0fefb1475e3b c0f4c53f35a12332 40f6012caf2450b4",
"c0fa8624bf56eead c102f2df30286b1b 410720ef5433ece4",
"40f392c665de2fca 4106766845078116 4108807dd862096c",
"c1030f4a622f074a 40fc7fab4f7429b0 4107cc23eb74fbbf",
"c0f9faa49b1278f1 c0f018eaeba5ed57 40fe8fdfe9139706",
"c0e477bfa7ad9906 40d42f364d0dfcd4 40e6d216657d43b2",
"4100b96f7648a73f 40e61f30f54f84c7 41019d7424875795",
"40ed71855c588eb8 c0e4a16b2e58e26c 40f1f9d633ef37d5",
"c0e66a178d590fe9 40e5f2d61ecf1825 40ef5f00fa7371dc",
"c0d26fe743924e7f c0f40332ad7fa7ee 40f4895408307d93",
"c0dffc867585f188 40e7d6b69818da81 40ecb4e24b4a00ef",
"c0f703d6700eb5b1 c0f8d3a952a584df 4100ed3fb5a2fba0",
"c0ec17b87e24676a 406a2e1ac943a9bd 40ec17c4b10af588",
"c0ad5e62e17eb917 4100b4f823ae62cd 4100b69512f000ae",
"40f43aa79dccfaa0 40f87d6e1abd26b4 40ffc3ae10ffd44a",
"c0c317d6a295798e 40e4fc468a1f64a8 40e5857e9883fc80",
"40e91121047b4efb 40f7cd04cb94d688 40fae633708a259c",
"c0e345121e40ee7b c0dcec17f8075b65 40e817af5a21220d",
"c0e9ba9ca0a13c84 c0c9c5e1a3ca15e0 40ea8602082b6a65",
"c0f01e170fc6ba6d 40e1190f24670d8d 40f23e937b2310ff",
"c0e509fcfb36c702 c0f29636d10cc32b 40f55b6ebd056d47",
"40e1d6f57665da9f 40f7983b2b7f3ad3 40f93973655ff0f7",
"c109af5029253346 40ed0da924afcb8f 410ab12aaeb4ea6b",
"c0d45366669536a4 c0fb119b28c62e3a 40fb8aa624c36c04",
"c0c6074ac8c7e77c c108ee5c6e0ecd09 4108f815e6a906e7",
"411007da37a61605 c0ec0f84fdd19f38 411068f0c1fa5da6",
"40f135190902674b 4104bc67db28fcec 41067335166838c1",
"c0f7a89646097e9a c0f660b053ddff24 410048562de221cc",
"c0f3929fe71b0570 40fe6b0e639d273d 410215eb4a16b339",
"40e68564c0261139 c0dbe113fcd9c650 40ea7c6e21c61c66",
"c0e0ccc19e271e88 40f67662f6449c44 40f7fb4e34cb29a4",
"40f435d540fe8b15 40fd25517f420266 4101bbd19e20c01f",
"40d933dc94297f41 c1050577567432b0 4105418fc3a01691",
"4108dbeb73326c3f 40c31ffaa726c636 4108e345af1b060b",
"40d537385bea56ee 4108521ce8160c8b 4108770487d09aab",
"40f54eefc42b8a53 c09a07f05dd8ddb0 40f54fee256c25a8",
"40f1e335af8f70b6 40e5cae7542356f3 40f4f1ebec6cc090",
"c0d482da9f8ab213 40ca23db832cecc4 40d8524aa4c4c5c9",
"40cf202946416051 40c8834d5d17688d 40d3cf4472921f15",
"410105cea0700531 40e56d280428621c 4101d87848117343",
"c10347fb19826c61 c10ce528956c760c 41115e662e392b3b",
"40dbc746967a796e 40cf5bd791e25fa5 40dfe5e48c2f2ae6",
"40f8c229e14707e8 41051a008e0769fc 410876f68e05c763",
"c0e465f8a195fbbe 40ef8a392b61568e 40f2c7dc1f80e90f",
"40e8bf48aea510b8 40f5987b08d822a0 40f8e3a901138480",
"40d98787918cd293 c0f9559afad99329 40fa203f4059c389",
"c0f1d66cd9e0711a c0fd8faf183bf67f 41014355b44c3e9b",
"c0ee100554840703 40fc8d985189c304 4100224ba4c1101b",
"c0e553dcfe2d35f9 40f3af45565e3499 40f66335f7cede77",
"c0f3460856d89fbb 40ec972ccdade89f 40f7ff10dc43870c",
"410acc2a27d295b1 40e5837c129cfd73 410b54f921804ab1",
"40c970333e24658b 41100fc5276a83d4 411014cd9d2e8996",
"c10546a0b0f0112b c0f825d3b5e085f3 4108768c205ce585",
"c0eec1fdcd9496f9 40e806d346bd6e9b 40f383cedd0a4b35",
"c0ce1b9dc99aaf07 c1014fe2f26d1f56 410169fd87f5808c",
"c10de031678cc488 c105415273c04e48 41125527dc57d854",
"c0f471872bccfac4 40e2f460c8231ea6 40f6888dad6f62ec",
"40dfbaf5a98b3f9d 410606793883c7b0 4106612ade84478e",
"c0e9e28d1a626c0b 41007c580a44f7db 4101b5d8591a4079",
"40e5e14e95b69bdc 40f2902a736cd325 40f58c0b3affe8e1",
"c110f433ace76f7a 40c20e5fd411f7f7 4110f69addd8355e",
"c0e642d1d5c07993 40f7527c3b58f350 40f9d7932acfdf47",
"4109479e0148db98 40de552e9c1eb3fc 41099000fc1a5a97",
"c10bc3e5112177e2 c0f6c1d876cfa6af 410e019ebf870efa",
"40eda6c48a68dbd9 40fbb857a8dc08a3 40ff6f8c0f457f6c",
"40b39a9e7215874b c0f0996f68723c41 40f0a4fee5e9719c",
"40b40c38ac024c8c c0ac895e2c632824 40b89b594f5951cc",
"40c639515295a2ee 40e17d4479db1bae 40e259c2740169a0",
"c0fd855ebc3c86c9 40fc5e3224f3104e 4104788c07c4c171",
"c0dbaefc53f0f7d5 c1023aae971641fe 41028e052da146c0",
"40f1adce09818c5d 40f7146c40cdb2c5 40fd129ae56477b5",
"c0f96cc881657320 c0f8006287c2a801 41017b6cdd2b953f",
"c107707226388745 40fc32f117330351 410b5a66dc6d09fb",
"40e2369c537797a2 40bb7809bb41e88c 40e288be3e12e802",
"c0e998f25101bec6 40d65bc86738e625 40ebeea1aa55d17d",
"40e326f84cc315a9 40fb4c58343de020 40fceddbbd5d1ffb",
"40eed9920e9d4a8a c0ec5498542d42bf 40f4f13e803cb101",
"c0eb869ed91a8505 40e309a09b06dfad 40f0bbe5b5319b98",
"40e66324684a93f6 c0ec9b5a2091f778 40f229a9d047b1a9",
"c0fdff06a48153c9 41018e438ea7e90b 4107170782b78865",
"c1000036af3b2569 40cd29dcc3515739 41001ab47d38c499",
"40f4adbbbd29e3ed c102a7bd1132a76f 4105542e75c8c7b1",
"409ea585d7a5f0a4 40efe3e91a9f6b3b 40efe79755e9ee67",
"40e31103112418ef c0d46eaf448af807 40e5a188f2f3a869",
"c0dfb009d822bf38 40ea01ce929bb763 40ee740122159c43",
"c0b47a8ea0a83421 c0f66276cee8c559 40f66bd2f28227ca",
"c0f98c9a66690c5e 41047fe25a69fd21 4108277569b4129c",
"c0ed732558c61b7f 40eb6a7ad66c158e 40f41e30570ed513",
"40f26f79d481ade8 c1092d39f71f4ca2 410acf9f3b0aeb2b",
"40f5b62fbcb1e980 c0f71e6270a4128e 40ffb72838f93263",
"c0b4189d73e743b5 4103fc0254d87bc9 4103fe88d82f8689",
"40a170bcdab1a809 c0f78b2959192776 40f78cc6b5dbe5dd",
"c10399c6bbb847ed 40f127bd9998de75 410565380140f1a2",
"c102a0847a3ea054 c0f88b9ea2223edc 41064e7e286bff43",
"40e4fe0946fdc7ce c0f88ebd14ba37e1 40fab4e5682fb74e",
"40f7ba938c01c511 c106b31c81df6d1b 41099cfdc4a9c379",
"c12d9113bc613761 41767a84970ac7b0 41767f608abcb94a",
"41471c57b9115a70 c1492040d59d1756 415111b252994663",
"c15ae3eab3e1c42a c15d0db060b0acae 4163cb3c72e0f020",
"4172fc92f50caed2 4127e11e3349535f 41730053a40e1040",
"41704375e7e8f049 c16c269ddd40e8d8 417582331db546f1",
"4170343d5928d760 c159384b60715958 41716331992fc82f",
"416013b4d17271b2 c17303f10032e922 4174a50966de8698",
"414b3c0de163142c c15acac22e88daa1 415e0dd35f4b4e8f",
"412b79fb08e3ea37 c1505b3e2ddc0c2e 4150b68f77e7278e",
"416f8612d0cea82d 415b224b03d3abc2 417128dbd8e2232e",
"c16083bbbc5b5d6e 414da39bbb210751 416219c90d1330cd",
"4172263cd9d35ba4 c15157b0783df754 4172a8f8963c7b1d",
"c172f6c850a90165 c1746a45eb383eff 417bdd389bdcd260",
"416afeeff375a418 c137b81a0e248ff1 416b287e21d66443",
"410771fe23ce2251 c130acc4d4c6761f 4130ee325d8aa9c2",
"416ae77cc153e75d 416511c1fa380359 417116130588cf59",
"c149c1e218fa1ace c152bd590836be40 4156bd0739f1e5e6",
"41698d6f1c65c5c5 c15d7717a5571c7f 416d7ed28c652f8f",
"416cef9f9898c438 415c444a538051ef 41701a02825ea33b",
"415da40b89727bab c15f0c7bdde9aa34 4165767871840408",
"417030c5d5b7114d 41635b76b7c75f51 4172dce6f863178c",
"417c6076a95700b3 414370e3eb453eb7 417c7b0d8c499206",
"c168dcfe6d740e6b c14455f0149509cc 416960b12f38ba37",
"c17064a0f5cee510 4174777ca23add0c 417a38f7597e4fbf",
"4154d9645e0b4cdb c13bd00f9a710d40 4155fa628fd35ffc",
"4152315fc2d28104 414c1a7965317cd3 4156fcd4ff859890",
"c16ab3bd2d738454 c179583b9d2e06fb 417ca56a9fc80d34",
"c16f33a78dd2f7b1 c14a7a254096446e 416fe56b6e3f21a8",
"4158f1e028375ac7 c1168ae8f791905c 4158fc0da9f335a9",
"417e73f327b476b9 c17b5c3bdf581c1b 41847822f777e352",
"415a100ff1438b1f 416a0c3aff1628ed 416d202a51d7463a",
"4167e4312ddf291e 415337292de83e44 4169c037e0c32e61",
"4153c880dd54d9de c15c509c03852136 4161454c7fa01edd",
"c1640aca8c27c918 414d2ccff7279253 416553fcda33c340",
"c145c0dadaa7b345 4152e3c1fe63997a 4155cc1b49c506f4",
"415c630756cb6c62 416eb5c3050644bd 4170ea6861f606aa",
"c166b9fc20250eb8 c1636ba5f3982a04 416de4dba0067ba0",
"c15a614144706aab c162e01e0f258d54 416706fed89528c6",
"c16ca9697b64b393 4160ecd8213a9c2f 4170a49be4b34cfc",
"c15d15292783cbfa 4160fb12cedd4f3c 41665b2b04bf0389",
"416f7c9dd615bd5c 4164705ad7444f30 4172c4f92ed5aaea",
"416aee345723bc94 4146e7cb6875c6c0 416b8856619754ae",
"4152f48c76e5ae69 c15f37f3bd093340 416242e4ce634d4a",
"c167cf3a8e5f48c2 40ea071016b9e199 4167cf48c841009f",
"417e101e7add9734 4147366713307903 417e33e1d1e86704",
"c17625ace312fd30 c165075f1b89a3bb 4178842d2b010e1f",
"c178d4a1e7b46a01 c16bf23d18b8688c 417c7e0227343ee0",
"c16144c02e334fd0 4159da3f30fe34f9 41659213b78bbde0",
"41679bca5840ab26 4150ec0bb7f87727 41691434fc9d190d",
"c15b01c549227b10 c14d88acb57a2795 415ec7cb9c5c2ab3",
"4104f01ce99c6425 415c12960e15ef77 415c1489b78c1dee",
"41685692ac9da3d4 c161f99ba84e2143 416e41a22243e55d",
"c14ee77d938f7a48 c15c9a89c4c52f45 4160415ac1b183ce",
"4145ac67599617fd c0deb6a093106e14 4145acbe651e05d1",
"c15fccc69d35ffb7 c1441d6ec36a3a25 4160ad1d9e7ade20",
"4149f1a172a1dd28 41552299b2e22e3f 4158cc6b01642062",
"c1633a852099ec1a c13dafff45c285e5 4163955a39f7d9d3",
"415957704e86dbf6 c150d7bbf80be5a4 415e6d9a56cd7b55",
"c160a8e11b5c746b 416155e7c4892700 41680b03a0e07b1a",
"c16eac6c51322711 4139641632c3a464 416ed658f67aae4e",
"4161a26cc599b84e c12b074ce283ecc8 4161b7174cc339b2",
"c143eb756c4aca91 c16ef5870784514a 416f5b683a759f27",
"411f03a76bee3073 c16d306821b36558 416d34865b1c9e90",
"c11f7ae1804b9506 4160e90f737998b0 4160f061255cdc2f",
"c16c5e21c26d9607 4107b17f3dc0aea9 416c5ec01106d777",
"c1666f586e54b82d 4172034ca1bf9943 4175385d5a816944",
"41218d001b4db581 c14292e269533982 414315c1aa68868a",
"417047b885a513f5 4152c13a476257ab 4170f11f68df0ce1",
"c163745de0cfa2ca c142adb187949e06 416401d328354bc4",
"4148b232f40eb092 414473b29052efc9 415008596492eb6a",
"c1685051b8c1f5e2 c12003576771ca71 4168559710a9af8a",
"c165caac40285543 4171988d59ab4769 4174b23e6c304cd5",
"41541e3fed381f73 4163167d1bff826e 4165937fb6e2d749",
"4167408e05b5c8b6 c170661750f9126d 41741a15e2337903",
"4166fd3358c5451f c158196fe797ef83 4169f4a0a2d034a6",
"c13c51e860780509 4156c67cefdff171 4157d9b51ad67c50",
"c15aac83b1085469 41488b10133b345c 415d5c80f619442d",
"c14ea64f52634fa4 c16330e0656fd76e 4164aa0104e1b33e",
"c15ac22a8693b13c 4149177280d396c8 415d8db55dd8a748",
"41684ca5d5d48841 41306fdbf168037f 416862d8606b4720",
"c14375fefd0097f8 c12275cce1d95cdf 4144002ab8e010a3",
"c141f4cbd367bec1 414587c3a6370595 414c090d66e90784",
"c15faac0aabfcbc5 4152254f542bc2fe 41623fadd81f453f",
"414a1eaab68e4c76 4171ca2595bda674 417216367a675903",
"c139802ba732deff c1558e33d83d33f7 41567a7cb4d5bb0e",
"c168f34fa3cb34cd 41668c41d73d19a6 4170d08d7e49f85c",
"4166d285e3ec8c51 41342eb847151054 4166f61c51335e25",
"c15816c894261991 c16bf9aab75c3b0e 416e753a9288e9ef",
"c15e17b171b26fff 4164089b548c4c13 41690dfeab30d07a",
"4154ca6e1977632e c172a1672683a6b5 4173578986438204",
"c16d64158c1cb315 c15c8c218a0a4cba 41705639b40e839b",
"c16bf06fadcbd47e c1422220244e9b8d 416c4dfa048eae90",
"414b1df3630082cc 4172aaa2c578b8e5 4172f8c8695f4f05",
"c166126b2e10b3cf 4121c715ad312d0b 41661992d67354db",
"4104f03da65a7e76 416887c4d23b209a 41688853cb74bec4",
"c144078d09ddba44 41738428ff0a0c68 4173ad1ad6ff574d",
"c17a0b0a52084dc6 4168a235b0fa49ff 417ccf0f21b3a888",
"c157966184ccf5ac 416923311343c94f 416bc43fbbe6fba2",
"c16018566af681cd 4171bc7ce62dfec8 41737a03543b521b",
"412c0ce6e43ccd34 41768e465af83e1f 417692a23520d143",
"408f00169b8dd131 408f00169b8dec42 4095eba96fb678de",
"c05e2cc4a81d1889 c05e2cc4a81d2f27 4065563c688e454f",
"40857cdd1d11ebb9 40857cdd1d11f72e 408e636565c9bf82",
"c072e81d0e15ad33 c072e81d0e15b14b 407abcf44327a72f",
"c0863b4dbf85ce4b c0863b4dbf85d84f 408f70b808ddd98a",
"4084fa7cc385d36f 4084fa7cc385e818 408dab041c6f0465",
"c05917f9915bae02 c05917f9915b84fd 4061be6faa8632c6",
"c0594a78b59c2ea8 c0594a78b59c3f67 4061e224891d6331",
"408511cad3b3bfbf 408511cad3b3ad36 408dcbf966080396",
"c03e8ce99df96c00 c03e8ce99df96b57 40459a38664d761e",
"40875b605a7323c5 40875b605a734a05 4090840ebdef55d5",
"c086892829af9900 c086892829afaa42 408fded1e5bc8b26",
"c08af32714c08df8 c08af32714c074f7 40930e70088f45f2",
"c0822fe5f9222716 c0822fe5f9221b8d 4089b86f3fcb398d",
"408e9b72e92e6474 408e9b72e92e1f1b 4095a47fc06e4440",
"408ed65d49b6dd6b 408ed65d49b6d8c2 4095ce289a50ca9b",
"40887c1945a10e24 40887c1945a132ac 4091503711649e88",
"c081ac34cfad5685 c081ac34cfad65a3 4088fe31a36b0375",
"c08e2d311fe0e99c c08e2d311fe0d39d 409556891b549320",
"c0803e7cc38b1c57 c0803e7cc38b0fe4 4086f8fd3da24056",
"c08b79d3591d9cab c08b79d3591daa40 40936daa6f870d17",
"4078fc89d26e07d4 4078fc89d26e058b 4081ab0920b899fc",
"c07709b3994a18ec c07709b3994a1c9a 40804a4e0512afd0",
"c08b833135efa8bf c08b833135efb0f9 40937449fb050078",
"408dbea55f3dc424 408dbea55f3dbe91 4095085e28ed234b",
"40805f734feaebac 40805f734feadec7 4087279b26778f45",
"405d10e035c1fc91 405d10e035c202a6 40648d7e666e8f1d",
"c073dbdc1e92b4b0 c073dbdc1e92ca77 407c15a9dd90b76f",
"40804bca6f9be3be 40804bca6f9bfef9 40870bcd96a65276",
"40842d62f786fbc8 40842d62f786f06e 408c88f5b377eeec",
"c088c3d8427d2629 c088c3d8427d146b 409182f27d50f6dd",
"4087688eb0b85908 4087688eb0b85e30 40908d60c22d3adc",
"c08747348d921717 c08747348d92106c 409075cb623b4895",
"c080f2de3fcd0792 c080f2de3fccefeb 4087f816111da1f7",
"c080c50afda0d085 c080c50afda0c987 4087b7478fdf6b43",
"4082c02561e54e0e 4082c02561e553e9 408a846e7dfd74c9",
"c08081a617bc18a9 c08081a617bc21b5 408757f84795f145",
"407ac4d89c7a79d7 407ac4d89c7a64f3 4082edb1a84df714",
"407b9d8db16bb244 407b9d8db16ba730 408386ede0d0bdae",
"4079830126e24e66 4079830126e28775 40820a1e190ac8dd",
"4071684821107b1f 40716848211092f4 40789e22540cde15",
"c087e707eb736652 c087e707eb734536 4090e6ceeabf18fe",
"c07422cc1695b00a c07422cc1695ad93 407c79fbf08cfcbc",
"c088294219cd4f88 c088294219cd65a5 409115a35558240a",
"40723f753219deda 40723f753219cec8 4079ce7060637f88",
"c073a873c9e36a0d c073a873c9e3491e 407bccf658e11b09",
"407bfcd722b6ce94 407bfcd722b6ec20 4083ca4ea596bb69",
"40879f05f7c3854f 40879f05f7c37ff1 4090b3e4250380fa",
"c07de8848dcaeb81 c07de8848dcac050 408525f9c41af68d",
"407e56cc3bcf75d8 407e56cc3bcf6e84 408573f494075f36",
"408373ad9b598ef9 408373ad9b599efc 408b8254109b2f37",
"c012c00bb37d7936 c012c00bb37d7993 401a844a2c605f28",
"4081e7e5a3485e1e 4081e7e5a3486832 4089529bfd94c42d",
"c08de320e1aa0514 c08de320e1aa19b5 4095222a30ba652d",
"c089dfa33aaf56af c089dfa33aaf710a 40924b9e7b93ab92",
"408889d33a17b077 408889d33a17a9df 409159ebcf2e8e67",
"c06092bcd831cae1 c06092bcd831b996 406770231cea7221",
"408f317c89864186 408f317c898649e3 40960e97757c99f7",
"408193cf399d39b1 408193cf399d314c 4088dbb10bb0b693",
"4088b75928d33059 4088b75928d353ff 40917a1c62504968",
"c08bb6cb5658c868 c08bb6cb5658ea31 409398c6f35da55e",
"c080fa9d8092f4b4 c080fa9d8092e7fc 4088030acf67cb7e",
"402e4c9441886f3b 402e4c9441887843 40356cbacd7e0e84",
"408008daabeb9ccf 408008daabeb8f54 4086ad23f5427828",
"4085497d425047a0 4085497d42503fd6 408e1abdddd83407",
"c0816696fca07f58 c0816696fca069bd 40889bbdc5c6455e",
"4086935ddce4112e 4086935ddce415c3 408fed423a38654e",
"4049242c4b94b7ca 4049242c4b94c8dc 4051c70fc49224e5",
"c08d911a47c37f82 c08d911a47c38d1f 4094e829efe31332",
"4019cc2b8e34fe00 4019cc2b8e350616 40223dda7e43e5ef",
"406eefa22c0e9d01 406eefa22c0ec3b0 4075e006cb6df1ff",
"c063f5f98a8bef92 c063f5f98a8c274d 406c3a987ac9122d",
"c0385a3cd3a9c82e c0385a3cd3a9d19d 404138458d33d87b",
"c086a95dfcc680ea c086a95dfcc67d29 4090062fa08e5e01",
"40622cd40990cf91 40622cd40990d948 4069b417c4a2fa55",
"c08bc378fcba51bf c08bc378fcba0985 4093a1bdf8c2776a",
"c0499316070a548c c0499316070a5ae7 4052157d2b21988a",
"c0878ecce94e95d4 c0878ecce94e7d28 4090a86b7d7d1f15",
"407e28d83db02be6 407e28d83db0226a 408553762de35800",
"406520338ebcaec3 406520338ebcafa7 406de05a0d293884",
"4031b5c45e6d963c 4031b5c45e6db2e0 40390bb701edc7fd",
"408aaab34e7e0105 408aaab34e7e109d 4092db34c6be92a1",
"c073a2912faf473d c073a2912faf560e 407bc4a3b0901d6c",
"408328057c50df7d 408328057c50bfdb 408b175567b7e984",
"40624934466fa5ca 40624934466f8be6 4069dc38f3a21c57",
"4074b4bc05df0c66 4074b4bc05df0dd6 407d485eddb2f924",
"c087f739c5077a96 c087f739c50779a3 4090f24279bed845",
"c07b3e259fbef225 c07b3e259fbeede6 40834377740e2dee",
"4060f88e1d463333 4060f88e1d461a7b 40680020f8981251",
"c08623f9935be793 c08623f9935bdc90 408f4fba1b75e430",
"408dae232256c206 408dae232256b587 4094fcb1c225dedf",
"40745b6885bbe58a 40745b6885bbee71 407cca0b562c7d78",
"c07650ea53cc438c c07650ea53cc706d 407f8f48487e3e43",
"c04bf9a12906143c c04bf9a129061f71 4053c8096c27f784",
"c051d12ac42db178 c051d12ac42dc2ba 40593276dd1219e8",
"408c64d28f3dd239 408c64d28f3dbdaa 409413d56c0895fa",
"40101c46bc24c0fa 40101c46bc24cbce 4016c89b847ae20c",
"40617e1ea7fffdea 40617e1ea7fff059 4068bd04870dcc58",
"407463ccbc78e63f 407463ccbc78d71a 407cd5e95ea70d99",
"4058246c971b8a1e 4058246c971bad3f 406112384802c6f9",
"401c24a1c8886cdd 3c60393004257e9c 401c24a1c8886cdd",
"4021bc5569691386 3c998f4f74059f64 4021bc5569691386",
"401077a854f97038 3bb370f8504dbede 401077a854f97038",
"3ffe51fcc13e693c 3cab4f606bb8c52d 3ffe51fcc13e693c",
"400328b2b8a6739f 3c7b9c6a392861d4 400328b2b8a6739f",
"3ff447ddedf0a55c 3ac39d43fc5c4605 3ff447ddedf0a55c",
"4001b288ea3d7da8 3b3abe59c4a997bc 4001b288ea3d7da8",
"40223d9f872faa86 3c9a49a2dda3eec6 40223d9f872faa86",
"4020a2528e4ba37b 3af01665949ffd82 4020a2528e4ba37b",
"400c15266e58edd8 3bb093b9a67c66e4 400c15266e58edd8",
"401d6e82ba1e2cd0 3aec76ee7b8c9575 401d6e82ba1e2cd0",
"4023fa46da9ac6e3 3b5e3079ff281ee7 4023fa46da9ac6e3",
"401e19c92369e9b8 3bf635ddc38ffdc6 401e19c92369e9b8",
"4018dc94764f83ce 3cfbfde1447121aa 4018dc94764f83ce",
"400ad99cec5f3b2e 3bafb2ee0d4a91f8 400ad99cec5f3b2e",
"3ff5bc3e3206e078 3b64874764dbbf92 3ff5bc3e3206e078",
"400aa500c869f77e 3b4421cddfbf428e 400aa500c869f77e",
"401ba6718f128034 3cc8e7b1f77747ca 401ba6718f128034",
"401a4acc6e3ee46f 3cfd9a3499962ab6 401a4acc6e3ee46f",
"4010feffc79c9ea2 3b800d670a4a14e2 4010feffc79c9ea2",
"3ffe191c996da1fa 3b6c6d45b307362c 3ffe191c996da1fa",
"4017061d2a306a53 3b51657770f475e4 4017061d2a306a53",
"401c77d8afa8536e 3bc0cdfc37020f61 401c77d8afa8536e",
"402033f8c3c801fe 3c62ae492b45b56d 402033f8c3c801fe",
"402162643339d21d 3bf9a79e564f8a15 402162643339d21d",
"401a97869b8251fe 3c2886d595fe39c7 401a97869b8251fe",
"400d6fdbcda350bc 3c1b269a4e1ece09 400d6fdbcda350bc",
"40208ee9d3579c6e 3b240488263d64d9 40208ee9d3579c6e",
"401540468563a70d 3c8ea043630319f3 401540468563a70d",
"401690d757fd53fd 3cc45350e2f11c5b 401690d757fd53fd",
"3ff22b1efa1d1c77 3ac19249c89e52f6 3ff22b1efa1d1c77",
"402238035ea15bf7 3d04833644acacb9 402238035ea15bf7",
"4022e88a11603c49 3cd107f8607f7b7b 4022e88a11603c49",
"4005659472d1087b 3ce81734e888d027 4005659472d1087b",
"3ffde44066a80ed8 3c0b91f4d0d75836 3ffde44066a80ed8",
"401e85694de00714 3c61981ffecd603b 401e85694de00714",
"4012b448a5fe2323 3c2140698ad61ab6 4012b448a5fe2323",
"4022a0756a6ed01a 3b2684b6794899d7 4022a0756a6ed01a",
"40180b2260805573 3bbc62b330a9231e 40180b2260805573",
"402258bdfdb5cc55 3bfb132b04810f52 402258bdfdb5cc55",
"4011ba7bd33886ce 3cbfefce2db89e65 4011ba7bd33886ce",
"4022811a008275ca 3b265ecded0e709f 4022811a008275ca",
"3ff80b1c137939cb 3c3bb85bb9d36831 3ff80b1c137939cb",
"401ad40d1bec3f58 3b54455a4f418eeb 401ad40d1bec3f58",
"40100a2136faf885 3b483d020171f0bd 40100a2136faf885",
"400e75a6398d5418 3add756c75c4eba8 400e75a6398d5418",
"4012e872d89083af 3cf549dc535c1fba 4012e872d89083af",
"4016008bd2563262 3c244b1b5d407cd9 4016008bd2563262",
"400a96f9383c0d1b 3b10129021e956b2 400a96f9383c0d1b",
"40204f9fcf1450eb 3ccd6222cc7f8b8c 40204f9fcf1450eb",
"4018d4c5a2aa4d35 3b52c30a87d60ca0 4018d4c5a2aa4d35",
"401fcc73b88e8a68 3b8e087087978b53 401fcc73b88e8a68",
"4012278366578a1e 3cc05a1b99cc6d81 4012278366578a1e",
"400352c8cce72ac7 3ad2b03c7d354184 400352c8cce72ac7",
"40142aa002f7cfe9 3b4e7989d41c42cb 40142aa002f7cfe9",
"401f0910c49ec97a 3d0178ac9bdc9335 401f0910c49ec97a",
"400c3eab73d75530 3c504832834afe81 400c3eab73d75530",
"4023c2bfd4c5983d 3cd1cc842e20f648 4023c2bfd4c5983d",
"400db0795e519564 3b7c0a71e003c744 400db0795e519564",
"40185f01bafa491c 3b1d767f11742241 40185f01bafa491c",
"401c62d1d1c214ef 3cc9915e84dbde0e 401c62d1d1c214ef",
"4020875276f97cec 3c630e61dcb88ce3 4020875276f97cec",
"401a21fab40b351c 3c281a6ab4032123 401a21fab40b351c",
"40168b515315d1fe 3cc44e574052aa48 40168b515315d1fe",
"4007a2015e342b6b 3c4b3f2e674c9c42 4007a2015e342b6b",
"4021e075d24c4cc0 3bfa61a9cbbd6cd6 4021e075d24c4cc0",
"4003702e95566314 3ad2ccaafad0c238 4003702e95566314",
"4007c9c90905f6af 3c15f0d5e5e32938 4007c9c90905f6af",
"40199d2fe7209c94 3bf2e659a08f3673 40199d2fe7209c94",
"4011b253eb1f73ab 3cbfe11cf0bd16b6 4011b253eb1f73ab",
"4020074013339ab4 3bc2ec434e4e7c2b 4020074013339ab4",
"3ff0c63006a1b790 3b93cdae90fb7dbb 3ff0c63006a1b790",
"3ffae9a4afc5e800 3b9fc5daeba3bed8 3ffae9a4afc5e800",
"402029ff8a93a15a 3b8e887526732d68 402029ff8a93a15a",
"4020fdce4666386c 3af06edfbe3c3ab3 4020fdce4666386c",
"40216e214edec570 3c64187c5ac1ce10 40216e214edec570",
"400451dd295b4339 3b3eb4d5a146a763 400451dd295b4339",
"4011af097309dbd5 3cbfdb2f48737e50 4011af097309dbd5",
"400f96a1eabfba2d 3b131811c47c48c5 400f96a1eabfba2d",
"40206a381a1b2943 3aefc0462291bef7 40206a381a1b2943",
"4022f8ab1b5f53db 3c317f7c667b7fcb 4022f8ab1b5f53db",
"4022c8b512b6e61a 3cd0eb4c6a3c43c6 4022c8b512b6e61a",
"4019a0e435e2879d 3ae8c94e0b0e0e9f 4019a0e435e2879d",
"401ef209fce1dbda 3b5761b9339309a0 401ef209fce1dbda",
"3ff0c995e64ba830 3cd2e6a6b2c1333a 3ff0c995e64ba830",
"4019723456e880e4 3c5d565dcc3334fc 4019723456e880e4",
"40051a4bff1d0a64 3ce7c2720e74831b 40051a4bff1d0a64",
"4014f773b552e150 3beef0f607449bfc 4014f773b552e150",
"40226de9d1683734 3b26479b75bea593 40226de9d1683734",
"3ffc49ba6bc588e7 3bd4df73c8523c5d 3ffc49ba6bc588e7",
"3ff5bd012691a216 3bd00a3fa0ccb834 3ff5bd012691a216",
"4007493e91ddca1f 3b75fe3c7527f367 4007493e91ddca1f",
"3ffc3fc997325949 3c4048d775d30ffd 3ffc3fc997325949",
"40114b3c191743d4 3ae0b9c23be84b4c 40114b3c191743d4",
"401ccddffbc762dd 3d00371f5f8968c9 401ccddffbc762dd",
"401cdd0496698245 3bc109b4d1726203 401cdd0496698245",
"40233e44b39b67af 3c662f99c9d81ff2 40233e44b39b67af",
"3fff58045210b726 3ce1a51eb38de668 3fff58045210b726",
"4008b58788696237 3be23b6ab9cba8c7 4008b58788696237",
"3ff2b1efa9b7b899 3c358dcf09f2f040 3ff2b1efa9b7b899",
"0000000000000001 0000000000000002 0000000000000002",
"0000000000000002 0000000000000007 0000000000000007",
"0000000000000001 0000000000000006 0000000000000006",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000002 0000000000000001 0000000000000002",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000004 0000000000000004",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000001 0000000000000001",
"000000000000001e 000000000000003a 0000000000000041",
"000000000000001c 000000000000004e 0000000000000053",
"000000000000001b 0000000000000018 0000000000000024",
"000000000000000b 000000000000001a 000000000000001c",
"000000000000000a 000000000000000b 000000000000000f",
"0000000000000016 0000000000000028 000000000000002e",
"000000000000001e 0000000000000009 000000000000001f",
"0000000000000016 000000000000000b 0000000000000019",
"000000000000000b 0000000000000009 000000000000000e",
"0000000000000016 0000000000000003 0000000000000016",
"0000000000000013 0000000000000016 000000000000001d",
"0000000000000015 0000000000000039 000000000000003d",
"000c3e17cd984640 000a92b249fb1717 00102d1ab05bac93",
"001cea80a79198db 00018647b73ccd9b 001cf4c8a2d0909f",
"000f0e4d8afea41e 000d4ba8fb1f06a2 00141609adc0dd4d",
"001b3233a209a7f5 0006ad2844565c14 001c00ec89ffb1d5",
"0011682e3e84479a 0005535e934addaa 00123413d5c50c5b",
"000ea49fb1b1c313 00032f5f237e2419 000efc46950ae0a0",
"001077e5167440ea 002a656ac160e1c9 002ba691726eaf5b",
"0015735ef39bc46f 0002afaef0125494 00159e433b65e595",
"001dfd5730000ed1 00029f56d1ce0bee 001e1aa31575cf82",
"00177eb1aa6ae9c2 000526b6be9c6604 00180d8f7704f37d",
"000edd77b9dac626 0015941327677190 001a33fab647f455",
"001b0a36064d8503 000babaad1dff275 001d7370bce7a8c0",
"0170ac630965b393 0181cd86065b8d37 0183a8798ae7c3f2",
"0174c41a9e751198 01594bc5fd335e73 0175b526edddcd89",
"016870b314288d75 0188fd7ac31bb63f 0189b9ec6660817d",
"017af4c30c80bd6f 0149eb0e2bd9fc97 017b266c76257d16",
"0178fcaada4f8202 01776431dfa25c08 01811d17472840c1",
"0162a8b4d3802baf 01491ede444c854a 0163b001f940b2af",
"0171a7fd72159a97 013822610f9c177b 0171b87462ab2d1e",
"017daf4ee686bc59 015898033443482a 017e509a91643ab6",
"017ac4ce8228fb45 01810d0150f7c64f 0185ad30a42e05b4",
"0179151f3555d0ae 014c5f948771eb2c 017954ff04a914aa",
"0172981a23eea296 0181a256fc49cb7c 0183ef52b72cb590",
"0166d6f36cd83a2e 013aa09adf7c5b2b 016714b60debd307",
"1a6f82eb61078a61 1a66a3cbd9b61275 1a73668c98bc8697",
"1a73b31310ebe6b8 1a780c57e452d1ea 1a7f1636ad076762",
"1a7443d1c82379c2 1a55f404e16d5ae8 1a74feb5cc11b7bb",
"1a79318f2adba4fa 1a3ae9d7a14e4af0 1a793feb2a821af3",
"1a761ed06a9c24c6 1a4d39f12e689619 1a766b864d7cfa08",
"1a6b0ca6c2fcf4ef 1a88b5f39b8e5235 1a899e8d001d179f",
"1a75e5b75e9837bc 1a54b5c5e3b4b0ba 1a76804854566e48",
"1a75072b3de2d822 1a87a3b9cb0ecc23 1a89df4b08a3e62e",
"1a600aa65cc74481 1a432c248dd00981 1a60be0bc15752c8",
"1a714382f6ab5c78 1a633075cdbd5642 1a73c032f4c30ecc",
"1a7185ad1df3da31 1a626a78b89343df 1a73cb54f2a98778",
"1a7672ddec531fba 1a7a825046c643a1 1a815e5c9b3d1971",
"20b6ec1eb440c6ad 20d17f53631ea904 20d269712a58fbc4",
"20bc6926676a6a21 20c836bf76a94918 20cc12bc0ae84292",
"20b607816f95177e 20961c17c7823b71 20b6b651db906663",
"20b082293803108a 20719a90b2363a43 20b08b8957f6376b",
"20ac8059a33f3e55 207ed5da495e721a 20acc2c4d15dad97",
"20b3936ac2bf781b 20d3a67ae6fd5b32 20d44025793a54cd",
"20b2c432212dff8f 20837c2abebf9005 20b2ec7d3d1d7db2",
"20a4d9fa2542ea5d 20b8d90a45f769fb 20baf248e0a98238",
"20b9ffd1ac2fc70a 20c37a1da90f9acc 20c76ab12c22bf8a",
"20b3303f0f0402f8 20cd739efaa9b7f9 20cef9977fb43e40",
"20b65ad2db188a24 20c9ec0c1af4317c 20cc3aac43502418",
"20bdfa092b46c1fa 20be660fae03e47f 20c558b06199c9cb",
"2d395884d99aac44 2d1b8dd174d96d3a 2d3a43e2f3841f61",
"2d3573191ca9124a 2d2da17d431fdb3d 2d3a119c282f3cb1",
"2d34e451323a228f 2d4cd92e7594198b 2d4eae6d69951cfa",
"2d36dd81b0431be6 2d2a792f777e64d3 2d3a6b9619916c52",
"2d361670f62f63a5 2d4dac8e5175aa9f 2d4fa99af9d12d38",
"2d26e9ccf5c6ffce 2d0e0d2ceec204a4 2d281d0f7ef9c9e0",
"2d21304c31297409 2d12ca890461d259 2d2396c575c21f61",
"2d2488f58ab8d503 2d3d1117ddffba54 2d3ed3b0ad593392",
"2d3cb383a65f588d 2d1d7d71c8270ff3 2d3da20bd73006d3",
"2d267b598ae82a29 2d2872b94bff900a 2d309b57d9b2ba95",
"2d21781eb2151b2f 2d487638eeadf6de 2d48d93e5d794a1c",
"2d2dba8f86c9052a 2d23da58634efa56 2d31dfc66556a532",
"52b58e392ddc7e34 52c73e2bae832b42 52c99ec08311ab16",
"52b599e583d7a82d 528451b6eec20352 52b5bffdf0f188e9",
"52a511a315b82d47 5292ebfda9af770d 52a7187a12ed34ab",
"52b56d8d5907dcee 52a0a5aed004ec6f 52b6fce04c5edfb5",
"52a95b076feb9412 52d3c0efaca3c308 52d4019d6ad0be78",
"52b13c48debe7608 52b4791f4df40d4e 52bac31a7c4eb31c",
"52a47329d94bbb76 52ce29960d37938f 52ce97b81f3850ea",
"52b54fe530c88303 52b7a2c89e7d18e4 52bfd3512eccbed9",
"52b157d1a6e55c50 52a81367e6f7bdc6 52b51c864711544a",
"52b30d0b733112ed 528064f2962f7306 52b3292e0d77034e",
"52b3bcff51868df4 52d98da4b83fc85b 52da067fd618e413",
"52b21b9491a1fe6a 5294be098ed8dbee 52b2d5ead2660e35",
"5f2ca2ef7237eca1 5f4db1306e5d5f90 5f4e8b03c107b2fd",
"5f3063c6ef5e339b 5f09f03e6333463c 5f30b516f3f11c68",
"5f286a9d29946c08 5f17c876ac828df5 5f2b2884e2139026",
"5f3b2221fc67fd27 5f0267e0fb00a2f5 5f3b3b0f19fb8a84",
"5f242403f2fbf65e 5f5d6d52c26e0970 5f5d88d7bb52b09e",
"5f26186f8cc5e731 5f40c37e88c02e3f 5f41a679e85cb4cc",
"5f264695835d3b13 5f5ce144901b5df9 5f5d038d42cef1b7",
"5f3755b31c24dd13 5f01d9223294a39f 5f3770f0d7667d7d",
"5f3870b467361f5b 5f46b635b7c7944d 5f49ca65360ca82b",
"5f35421e72ead9e7 5f30c8b5d9300a41 5f3b15db7e2ae85d",
"5f37504fb20dbe68 5f108723ad50e9b9 5f37ad52f1dd6ab2",
"5f3544b47889215c 5f1acc2bf8251482 5f364c6e55b71ab7",
"656473d3767a6bee 657b4711499434d2 657d21a918d6fac8",
"6574325988b43232 6543fe877f746c69 657459c97c8cc9c6",
"656a214f6c4f60b3 655bb21cd81e8a0a 656d929ede7b1728",
"6560efbd94a2459a 656c915aa04ffe7b 65709afe69526563",
"6576b05f8a4fdb89 656dbfd2916d514b 657b215066afc6d2",
"6574978ce1661822 65467477ffdb02b8 6574c84c3fa035b5",
"65766a824d62f95c 6569b89a4abd3056 6579d7dead6a6fe2",
"656a22255ee73a7f 655d48141f412cf4 656df47f5344aac3",
"657e5b3f57c81836 657537710a1dc9c7 658284978308c116",
"6560c6c27a1f2902 659955aaee53659b 65996bd922b93db2",
"656fab14a259fe0d 6547934413904b84 65701b24ff8747cf",
"6575cbc17c5d9795 65867dca9ad66c66 6588fe178b128e2b",
"7e73d72d293fac46 7e64a1134a5a8194 7e765c8cca1bdc7d",
"7e72c423b15c0c76 7e50fd4ed6d74ef6 7e733da6997c255e",
"7e6ecc5ca1e44b12 7e8cbd9cd5bcb6ec 7e8dc11056d53528",
"7e7762a94a7f37bd 7e75576d1807a59d 7e7fa8e235cc17f1",
"7e79c4d79274edb6 7e42092522666fac 7e79de0a5a590a68",
"7e6d6e9e5f6b9690 7e65dee57ace4f63 7e72559005bf2bc2",
"7e7203ef0a98e278 7e69407a88fb367d 7e75ffcfd1685d05",
"7e7e142f696a813a 7e58e8eb53a99254 7e7eb77c8f08f424",
"7e681297e04d50ab 7e5bcdb4b6a23347 7e6bcc638b0b653d",
"7e7347cc357aa5ce 7e607d3e8ffec2dd 7e74f821675a3d47",
"7e77d77a65bf7227 7e97eef785c82a92 7e98aa1c7babef16",
"7e73831f5386946a 7e76bf321fd04fd2 7e7df8153c23c1c5",
"7fbb2b55d3ec06fc 7fc07c694bdddf93 7fc55ca45119b6ec",
"7fac80e5d06a76d1 7fa643e596099200 7fb2159ed2273337",
"7fb342fc313955fd 7fac53c636567cfe 7fb7e8981539f9aa",
"7fb51968313531a8 7f7decbfa2b174eb 7fb52e96227430f0",
"7fa645d786f90507 7fc2e07209448e2c 7fc3ae4ccb213617",
"7fb39ef508fea008 7fa998df0abb01a1 7fb76d17e880a240",
"7fb87858f669f885 7fb168cd41f8662d 7fbe07fc77fd805b",
"7fb9b99d7c2d62ee 7fc07426e3cd6a1d 7fc4e279dabd918e",
"7fb8c5d33b636d9e 7f7bb7bf227b935f 7fb8d55009c619d1",
"7fbc20c5dbc103dc 7f84e4c774b4ad72 7fbc3fbf04b59a0a",
"7fb15f85cb864f65 7fba17ba4070a5ff 7fbf58e98d962c3f",
"7fa18c89b023fe47 7f86bd6c4443d953 7fa272638c8e1ff9",
"5e5def9f9cabd41e 795540cd1301a6e7 795540cd1301a6e7",
"0d680ecf38ab0eb3 7a8dc81f37948a64 7a8dc81f37948a64",
"1b5c2ea4a8da4eea 73b9cd5bf408dbfb 73b9cd5bf408dbfb",
"10856498197bdff6 06bb755257cc8036 10856498197bdff6",
"5a5b17aa8625400d 25cc6d7e9238c43e 5a5b17aa8625400d",
"68427c13453901da 2d0dee6941963e6b 68427c13453901da",
"0715dbcaa0d679e8 74793ff4af78b759 74793ff4af78b759",
"74ac1f81282dc534 7b8a5999808272b0 7b8a5999808272b0",
"7bc47a2c698055a2 2c8b2645b04c1ba0 7bc47a2c698055a2",
"73657ca0bad92531 2f6b01eccba24fb5 73657ca0bad92531",
"20f0948bb5576f3b 4059b4dfc31e3bc8 4059b4dfc31e3bc8",
"059ebc298ae3fb92 0b36ec37fda749aa 0b36ec37fda749aa",
"15d5f53ab29fd191 6d0aac02209905fe 6d0aac02209905fe",
"1706b48b7b87dedf 4cf84cdad008c9ce 4cf84cdad008c9ce",
"6e5b42dcca722a3e 4bbefb1dc9da82de 6e5b42dcca722a3e",
"1b49fd6a26a54437 25bb15077e308123 25bb15077e308123",
"357e58e983c37220 415f217aa6f1773b 415f217aa6f1773b",
"53a6df8fe06fe123 65d96c67d0711eb0 65d96c67d0711eb0",
"703f5d42f1057187 6dfa2e4874ca0d7e 703f5d42f1057187",
"439ab8d82ba59dfe 0ddf4b989394c21c 439ab8d82ba59dfe",
"43ed32b891c1196b 2dab6c158381fcbc 43ed32b891c1196b",
"60495985c0054ab5 0d0c23f67bd16ddb 60495985c0054ab5",
"66c829e8d6d75598 3b7972046271c25d 66c829e8d6d75598",
"2e75f1c92f609e85 115edec3c63bbc5b 2e75f1c92f609e85",
"5968926c758caa8c 1a25b11d74d837be 5968926c758caa8c",
"6808d38b5a799b1c 63627723f50abd5e 6808d38b5a799b1c",
"506af65987c0515c 56dc076f7bfa6f69 56dc076f7bfa6f69",
"11ad72d539e00212 4189424d9283783f 4189424d9283783f",
"0b52a17fd01365f5 018c632e79bd973e 0b52a17fd01365f5",
"0c9cfe2701fecdfc 2a309b1beb244e2f 2a309b1beb244e2f",
"533828e588a77fdd 4f07399185a3a741 533828e588a77fdd",
"3c4d3b145247f29e 3beb187f9560f466 3c4d3bdd3e4f875d",
"70f0a1f2309a001b 617b8479b7ecb334 70f0a1f2309a001b",
"3ac93be2ef4acfb9 040898af843584dd 3ac93be2ef4acfb9",
"2e28e8f2a2287f6c 20679d6581a868a2 2e28e8f2a2287f6c",
"56e5646e7d8b1749 15494d9c0aa6b51c 56e5646e7d8b1749",
"150f5a88314d7158 49768cc482598b22 49768cc482598b22",
"0aafa0f3c31c16bb 33a4902dc66d0344 33a4902dc66d0344",
"30c9446642358da4 62cf78d85cd6b10b 62cf78d85cd6b10b",
"242b348042c1c880 69ca1eb030dc6b82 69ca1eb030dc6b82",
"5bf9adfc2d003c7d 5c3586692eb27d61 5c3595b52af85c4f",
"76cab7c477bb1e90 3ab4ed2792275418 76cab7c477bb1e90",
"1c927b7d4adcca4d 4c6bfdd4ab6a1b03 4c6bfdd4ab6a1b03",
"21a9f79f9f5d11de 0cc9e5dbcb7d1a92 21a9f79f9f5d11de",
"10c8bfd8993365f2 780c092edafd3c42 780c092edafd3c42",
"22653fc9f30febd1 2cb13fde1e8515d3 2cb13fde1e8515d3",
"49d87db9f6e86025 0124541d737c6d65 49d87db9f6e86025",
"30d345ae38b9cef3 7e74a3167da9ba55 7e74a3167da9ba55",
"6befc675b3cbede3 56c03e056201c848 6befc675b3cbede3",
"60175ecaae9ab920 25bd2afecf7c4ea7 60175ecaae9ab920",
"05a0dc57b4a9dd10 0ede6a259dae3b6e 0ede6a259dae3b6e",
"4f6ae83014fc6c20 4069bb327e4fe36c 4f6ae83014fc6c20",
"2e3a88eb08d2d0ab 33848cc573081cd7 33848cc573081cd7",
"03caf1d686234b8d 05c7bb31d621468f 05c7bb31d621468f",
"3d5afc345232681f 31d8351448070308 3d5afc345232681f",
"101be07534b85ae3 2924c62d67fdddfb 2924c62d67fdddfb",
"5b3909a8e506ee7e 22412e6eebe4d398 5b3909a8e506ee7e",
"4c7d6454a8c8eb10 407876f14ad41a56 4c7d6454a8c8eb10",
"30181a897cdd027b 62ee7949ee981413 62ee7949ee981413",
"0ff21ba37a277808 2c911f4d39d28246 2c911f4d39d28246",
"63a6902dce75432c 3c5cc8f7877ff9f6 63a6902dce75432c",
"0e11d38b7dde7171 025962130f920c83 0e11d38b7dde7171",
"129336a64a76d912 5bf593315ce92897 5bf593315ce92897",
"7b9d9550bf8dceec 0db09a1b4959b52a 7b9d9550bf8dceec",
"62380559d3c535cb 32019965ecced5e2 62380559d3c535cb",
"7a7748ef2e39dc99 2d56e9c9187e509d 7a7748ef2e39dc99",
"21bc8cf6ee5bc411 74a09f1be52e9a7d 74a09f1be52e9a7d",
"0b40e04d2b5a8044 54739d03c80b329b 54739d03c80b329b",
"1415e6a6997fdcc1 21cbf52971ef88de 21cbf52971ef88de",
"33e498813a7230d1 106ee73a85d5ca33 33e498813a7230d1",
"7b0aa2cc590aa82e 1eb907193cf254d6 7b0aa2cc590aa82e",
"158ccc72d03f443e 39efff500ed23bd0 39efff500ed23bd0",
"380200eacf9a5580 38672dc1d5ddbdb7 38672e31b3d2ed56",
"287319ad210f4b83 54f522d17586a593 54f522d17586a593",
"1c3b69fd6c540c27 5bb112c60a2b1050 5bb112c60a2b1050",
"5b9bf5de09460d98 0b21cd60fc0ac5df 5b9bf5de09460d98",
"2b401de1541d722c 5d3c3ea4ae38edca 5d3c3ea4ae38edca",
"44ca41c897f2b3e4 3d886a281778df75 44ca41c897f2b3e4",
"31069a40d3c73cb2 2fc9fce91eddc2ca 31069a40d3c74ba3",
"2d5cec5d63676006 50b1178aa438d717 50b1178aa438d717",
"2ad010854c7d17f1 73db42094a4eadb2 73db42094a4eadb2",
"397a7be8ace20946 2328b28ab3989684 397a7be8ace20946",
"3b035098f6e5ad9c 593b6c00be50fe35 593b6c00be50fe35",
"75aacc8411eacf71 0e8fee1d19dec40b 75aacc8411eacf71",
"2adbb15c6740bc1f 1fc11d4e0fa85085 2adbb15c6740bc1f",
"5d26c466799d0311 19575438c0ce4d26 5d26c466799d0311",
"794455f01424f65c 76ce658083b90bfa 794455f01424f65c",
"444f872136009098 20ee6db9afcb2abc 444f872136009098",
"45cf317e38ee0cdd 39d9cd2f1a6cb067 45cf317e38ee0cdd",
"2dc39e3412072868 4ddbf9224fcc7e87 4ddbf9224fcc7e87",
"4cfe8998b5b0a434 04ab5480928888a5 4cfe8998b5b0a434",
"352daaabad743d27 7d6c63abccdfa030 7d6c63abccdfa030",
"1410ef79cbf06140 320cf8d58ce82500 320cf8d58ce82500",
"5e79577b4c1c2eeb 38df693d3f59f004 5e79577b4c1c2eeb",
"2af7f87bf4f5cd00 68f9e5db3db8b0d3 68f9e5db3db8b0d3",
"31d2a6eefca99379 18188a495eae8897 31d2a6eefca99379",
"0d649aeb4e3484a2 5e16fb187c7531f7 5e16fb187c7531f7",
"15b6c00600220903 18ff2d4021571e18 18ff2d4021571e18",
"36b428db1e7437bf 650ff795f1212c1b 650ff795f1212c1b",
"542a31624173ab99 2901524b90408364 542a31624173ab99",
"7c8c5e286a3967d1 7bb31cdeb59491b3 7c8c5e286bd579e7",
"6afbabd2bdd30b46 65b571c5416caeb0 6afbabd2bdd30b46",
"636814754a0887ca 19733695a7ea5beb 636814754a0887ca",
"413c2d3a6fed076f 536ab03a8bffa1f1 536ab03a8bffa1f1",
"21bd6064f57cf162 2f8b21cc30a8afd8 2f8b21cc30a8afd8",
"5d8d89990b5be3df 7f8e6eb89a8bc629 7f8e6eb89a8bc629",
"2fe544d12e2393fc 05d1fda3943bea7c 2fe544d12e2393fc",
"2affdfa202256fea 54e8031f5ca5c78e 54e8031f5ca5c78e",
"76e38ea8be83540a 4bc63d24b22eec98 76e38ea8be83540a",
"65333962a18bafed 6de04e95feba4680 6de04e95feba4680",
"6c4b10e7c8a126f3 0419d77b5dc01167 6c4b10e7c8a126f3",
"4691e87c94239160 7b766f3da4adf252 7b766f3da4adf252",
"013d93bf212f547a 500f000ca556db57 500f000ca556db57",
"7cd8e6acf9d759c9 192d31a2b8cc7033 7cd8e6acf9d759c9",
"3adfa8829177c490 0e20119242fd98c5 3adfa8829177c490",
"17a5a34b15213c85 08701d9a81dff8f7 17a5a34b15213c85",
"5095d5c47ab00539 75b063ef6b8e94d7 75b063ef6b8e94d7",
"0e4d324169905f76 45c20092bc93ec8b 45c20092bc93ec8b",
"6b0d586161362a64 735e1463203ba963 735e1463203ba963",
"1f9aee18be444080 7e0a4f400c8d019f 7e0a4f400c8d019f",
"10fa62d7493bae48 053dfb8e24953e9d 10fa62d7493bae48",
"535debfb4626df73 4cbb2be4eef10c32 535debfb4626df73",
"44a503e418446bf2 385d6cea9cede8d8 44a503e418446bf2",
"33c3ae701ddfa135 40de6104003c348a 40de6104003c348a",
"18577aa710b93df2 6ade379633ab5242 6ade379633ab5242",
"0e413de2f79f951b 25da1ada55a8a167 25da1ada55a8a167",
"359db4204791faed 227a7ffd255f03e5 359db4204791faed",
"0d10c8f0867a8bcf 395d6325b276dbbf 395d6325b276dbbf",
"522d8d0c4bc44151 085049b5d4eb5cdc 522d8d0c4bc44151",
"3ce616157b3fb406 0f381528e13a4f33 3ce616157b3fb406",
"1b83524c6b5ab3d3 126102916fda4b59 1b83524c6b5ab3d3",
"1e0416e2985c7305 265d2e4bde2cb6bd 265d2e4bde2cb6bd",
"0a00410a1078d793 1bfeacfbba2b164d 1bfeacfbba2b164d",
"10d0f06221250ff9 68d66cebcae4d283 68d66cebcae4d283",
"7bc643fec66b131d 7c5f1408ff2ff3e9 7c5f140afda5d340",
"4023692f99c86d89 3b122b226c53c2b2 4023692f99c86d89",
"2b79851a625f2388 0284f0628adbc394 2b79851a625f2388",
"5a6a5be18ff2278f 6efe8f106f951869 6efe8f106f951869",
"5f3834f356590f2e 0e565538b6517fb8 5f3834f356590f2e",
"0393f6c155a8a4b3 4c3eeb023d652e17 4c3eeb023d652e17",
"1e57de62a95410fd 46a3e965f0b67ac0 46a3e965f0b67ac0",
"3ee97ee4f034dc65 715f1bb809a743da 715f1bb809a743da",
"66661e13543db7ab 6f25b3a0f3969dd0 6f25b3a0f3969dd0",
"028d20d406fe0544 2a2723e4bf87728f 2a2723e4bf87728f",
"25e337d7d442f424 35306ebfed28072d 35306ebfed28072d",
"1452a096e4c815f9 552e984206234c68 552e984206234c68",
"33e46ac3e5bcc2e7 240e6acc8500c77b 33e46ac3e5bcc2e7",
"51cf51ee73aa72c0 3f1e7926788934f6 51cf51ee73aa72c0",
"39aa9a8b37f1bc27 63f7c2f30b34e72d 63f7c2f30b34e72d",
"0b36554c7fcbb798 565c365efc8d434b 565c365efc8d434b",
"4e20794486f8463f 0d1515ddf4aea93f 4e20794486f8463f",
"528d18c99fc9c87d 245fb5cb8b0b6aa5 528d18c99fc9c87d",
"16dda97e0bdafd23 7386b7dec59c01d7 7386b7dec59c01d7",
"17b1ca3bb946b51f 53459f990af8171e 53459f990af8171e",
"020609a2b5d2053c 44c44f9476c7359d 44c44f9476c7359d",
"64c80a3a8e4c69c7 52bc6fc3aecd1e2c 64c80a3a8e4c69c7",
"505814f49ae10d80 4a4c6ace35f06b78 505814f49ae10d80",
"155ab2961dca16f6 09d6bfc0e4713695 155ab2961dca16f6",
"5b829de4657129c5 57016345300c3092 5b829de4657129c5",
"75155cffc71e4595 6130610dab1a4c1e 75155cffc71e4595",
"6d883cc29b2ab04e 261920813530e868 6d883cc29b2ab04e",
"466fd221cb688242 6b23e2b6e3d62877 6b23e2b6e3d62877",
"2d0e23bf6b8803b0 45cf80ddb14de75b 45cf80ddb14de75b",
"34082345e4f0a01a 095ed5650c3de982 34082345e4f0a01a",
"590ac38e8c28fce3 5fce02bb39250f42 5fce02bb39250f42",
"7188994ff2bf3a8a 18bc0caeb4e2f7b4 7188994ff2bf3a8a",
"66f842256cd912a5 03092418d552a6a7 66f842256cd912a5",
"0ba7012c400ce2ab 4b71af9ddc5cb219 4b71af9ddc5cb219",
"424b117feb1e3397 4dbbae39137f9d5c 4dbbae39137f9d5c",
"0358402aeed7189f 67c492328f654c1a 67c492328f654c1a",
"19b4bce8bb3357af 5b2854ed154c4ed9 5b2854ed154c4ed9",
"23b8f74f7b1c131b 69ac0c48785e5f10 69ac0c48785e5f10",
"7bc7b268a4cc94a3 7719330a464e2ca4 7bc7b268a4cc94a3",
"7238412328ae2416 1e718a21e4aae70c 7238412328ae2416",
"0c236bdd3fd7f7df 60ad2617c4dcdfd2 60ad2617c4dcdfd2",
"112d6361d0b9d45c 787019cf4e8f4a56 787019cf4e8f4a56",
"5d23fbb847563a67 4216e5293926f416 5d23fbb847563a67",
"47cf6a6434a4f1b8 3bf26ab702d03b8b 47cf6a6434a4f1b8",
"421b82d814cb1942 07a089547c378329 421b82d814cb1942",
"764e6b4f1c516836 29175aa5fcaf9599 764e6b4f1c516836",
"52651bc32dc4f8bf 13ee722d59248c45 52651bc32dc4f8bf",
"445df5b398898de9 2f09262b6d09daec 445df5b398898de9",
"18b17c4bde506db1 7664049c6751949a 7664049c6751949a",
"032e674148c77c59 4f076704d30a6926 4f076704d30a6926",
"26e834766ca2c95a 4b03437f09875625 4b03437f09875625",
"46d40de2e2f601ec 50cd3d7320de946e 50cd3d7320de946e",
"7c1b4ac49e5f35a1 0e1fc0eb434a9619 7c1b4ac49e5f35a1",
"69ade49b9c1dad61 0cb29a1f62cf20f9 69ade49b9c1dad61",
"755c8595ba7bbfa9 6aeb2284edaa0923 755c8595ba7bbfa9",
"27e8bb73d38485a9 52df76d1cd2f0d49 52df76d1cd2f0d49",
"381aa8d04f8261a9 6b0d347cd590c2af 6b0d347cd590c2af",
"5e89e4000a61a41a 0244184baa311e6b 5e89e4000a61a41a",
"78b3000cf413e244 58065e688e7c5880 78b3000cf413e244",
"7a74986cff2d6b3f 4c91464072c00d05 7a74986cff2d6b3f",
"4485894e2b760312 008e02252ba55b26 4485894e2b760312",
"1cd6e9bce7a9297d 008aa9766f6515b6 1cd6e9bce7a9297d",
"1ea4e23051c6345a 0888c387871e6c40 1ea4e23051c6345a",
"550c3d20dd07e2f6 3ac9803840a59d94 550c3d20dd07e2f6",
"2837cd32949ae3a2 24c473dfcdc4db01 2837cd32949ae3a2",
"7fec265471126973 5836dba91c6f4cd6 7fec265471126973",
"45c31b65cfdb54d3 727826c922cbfd05 727826c922cbfd05",
"0d91a27fb5e74e4b 5cbb923fa815421a 5cbb923fa815421a",
"1277566ea89dffd8 23d62fcc0fe1daa6 23d62fcc0fe1daa6",
"35f906c1ccb309e3 09d24f667a6bb6f2 35f906c1ccb309e3",
"762a5d2adf3d4985 2f68298dc3dec574 762a5d2adf3d4985",
"16867312a5a58e13 49e41110580b7226 49e41110580b7226",
"0000000000000000 0000000000000000 0000000000000000",
"8000000000000000 0000000000000000 0000000000000000",
"0000000000000000 c008000000000000 4008000000000000",
"4008000000000000 4010000000000000 4014000000000000",
"c008000000000000 c010000000000000 4014000000000000",
"7ff0000000000000 3ff0000000000000 7ff0000000000000",
"3ff0000000000000 fff0000000000000 7ff0000000000000",
"7ff8000000000000 3ff0000000000000 7ff8000000000000",
"7ff8000000000000 7ff0000000000000 7ff0000000000000",
"7ff0000000000000 7ff8000000000000 7ff0000000000000",
"0000000000000001 0000000000000001 0000000000000001",
"0000000000000001 0000000000000000 0000000000000001",
"7fefffffffffffff 7fefffffffffffff 7ff0000000000000",
"0010000000000000 000012688b70e62b 0010000a96eeeaa3"
]


In [ ]:
if MODE == "java_hypot":
    import math

    from pyspark.sql import functions as F

    def f(h):
        return struct.unpack("<d", struct.pack("<Q", int(h, 16)))[0]

    def bits(v):
        return struct.unpack("<Q", struct.pack("<d", v))[0]

    rows = [(i, f(x), f(y), int(w, 16)) for i, (x, y, w) in enumerate(l.split() for l in HYPOT_VECTORS)]

    def same(got, want):
        return bits(got) == want or (math.isnan(got) and math.isnan(struct.unpack("<d", struct.pack("<Q", want))[0]))

    jm = spark._jvm.java.lang.Math
    RESULT["driver_java_mismatch"] = sum(1 for i, x, y, w in rows if not same(jm.hypot(x, y), w))
    df = spark.createDataFrame([(i, x, y) for i, x, y, w in rows], "i int, x double, y double").repartition(16)
    # Spark's hypot runs java.lang.Math.hypot in each executor's generated code; repeat it so the JIT compiles it
    got = {}
    for _ in range(5):
        got = {r.i: r.h for r in df.selectExpr("i", "hypot(x, y) AS h").collect()}
    RESULT["executor_java_mismatch"] = sum(1 for i, x, y, w in rows if not same(got[i], w))

    @F.udf("double")
    def py_hypot(x, y):
        from databricks.labs.gbx.pyvx._jmath import hypot

        return hypot(x, y)

    @F.udf("string")
    def arch(_):
        import platform as _p

        return _p.machine()

    pg = {r.i: r.h for r in df.select("i", py_hypot("x", "y").alias("h")).collect()}
    RESULT["executor_python_mismatch"] = sum(1 for i, x, y, w in rows if not same(pg[i], w))
    RESULT["executor_arch"] = sorted({r.a for r in df.select(arch("i").alias("a")).distinct().collect()})

    # the TIN of both levels on random point sets. A triangle is compared as its set of vertices: the levels may start
    # a triangle's ring on a different vertex, so its WKB bytes differ when the triangle does not (counted apart)
    import shapely

    from databricks.labs.gbx import register

    def tri_set(blobs):
        return frozenset(
            frozenset(map(tuple, shapely.get_coordinates(shapely.from_wkb(bytes(b)))[:-1].tolist())) for b in blobs
        )

    pts = spark.range(TIN_SETS).selectExpr(
        "id", "transform(sequence(1, 60), i -> concat('POINT Z (', cast(rand(41) * 100 AS string), ' ', "
        "cast(rand(42) * 100 AS string), ' ', cast(rand(43) * 10 AS string), ')')) AS points"
    ).cache()
    pts.count()
    pts.createOrReplaceTempView("tin_sets")
    tins, wkbs = {}, {}
    for level in ("L1", "L3"):
        register(spark, only=["gbx_st_triangulate"], _levels={"gbx_st_triangulate": (level, "-")})
        rows = spark.sql(
            "SELECT s.id, collect_list(t.triangle) AS tris FROM tin_sets s, "
            "LATERAL gbx_st_triangulate(s.points, CAST(NULL AS ARRAY<STRING>), 0D, 0D, 'NONENCROACHING') t GROUP BY s.id"
        ).collect()
        tins[level] = {r.id: tri_set(r.tris) for r in rows}
        wkbs[level] = {r.id: sorted(bytes(b) for b in r.tris) for r in rows}
    RESULT["tin_sets"] = len(tins["L1"])
    RESULT["tin_sets_differ"] = sum(1 for k in tins["L1"] if tins["L1"][k] != tins["L3"].get(k))
    RESULT["tin_sets_wkb_differ"] = sum(1 for k in wkbs["L1"] if wkbs["L1"][k] != wkbs["L3"].get(k))
    RESULT["verdict"] = (
        "PASS"
        if RESULT["driver_java_mismatch"] == 0
        and RESULT["executor_java_mismatch"] == 0
        and RESULT["executor_python_mismatch"] == 0
        and RESULT["tin_sets"] == TIN_SETS
        and RESULT["tin_sets_differ"] == 0
        else "FAIL"
    )


In [ ]:
print(json.dumps({k: v for k, v in RESULT.items() if not k.startswith("executors")})[:4000])
dbutils.notebook.exit(json.dumps(RESULT))
